# Alarm calls — clean bout analysis

Rewrite of `alarm_bouts_3.ipynb` using the shared `detect_bouts` helper. Thresholds come from `vocalization_analysis.bouts.BOUT_THRESHOLDS["alarm"]`.

We'll add analysis cells one at a time.

In [ ]:
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vocalization_analysis.bouts import (
    BOUT_THRESHOLDS,
    ALARM_SCALES,
    detect_alarm_scales,
    summarize_scale,
)
# Acoustic feature helpers - used in the per-bout feature trajectory section below.
from vocalization_analysis.acoustic_features import (
    SAT_PARAMS, FEATURE_COLUMNS, compute_features, load_call_slice,
)

HOST = platform.system()
if HOST == "Darwin":
    DROPBOX              = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR          = DROPBOX / "Data" / "parquet_cache"
    FIGURES_DIR          = DROPBOX / "Figures" / "alarm_bouts_clean"
    BASE_PROCESSED_AUDIO = None
    SAVE_FIGS            = True
elif HOST == "Linux":
    PARQUET_DIR          = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls/parquet_cache")
    BASE_PROCESSED_AUDIO = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
    FIGURES_DIR          = None
    SAVE_FIGS            = False
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

if SAVE_FIGS:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, fmt="pdf"):
    if not SAVE_FIGS:
        return
    fig.savefig(FIGURES_DIR / f"{name}.{fmt}", bbox_inches="tight")

CALL_TYPE     = "alarm"
DATES_TO_PLOT = ["2025_03", "2025_07", "2025_10", "2026_02"]

# Light/dark schedule per date (hour-of-day, 24h). Default = light from 8 to 20.
# 2026_02 ran on a shifted cycle so its light window starts earlier.
LIGHT_DEFAULT   = (8, 20)
LIGHT_OVERRIDES = {"2026_02": (4, 16)}

print(f"HOST              = {HOST}")
print(f"PARQUET_DIR       = {PARQUET_DIR}")
print(f"SAVE_FIGS         = {SAVE_FIGS}")
print(f"call_type         = {CALL_TYPE}")
print(f"alarm scales:")
for s in ALARM_SCALES:
    print(f"  {s['name']:8s} -> max_icg_s={s['max_icg_s']}, min_bout_size={s['min_bout_size']}")

In [ ]:
# Load + filter to alarm calls, then attach BOTH scales' bout columns
# (bout_* at 2 s, event_* at 30 s - defined by ALARM_SCALES).
parts = []
for date_tag in DATES_TO_PLOT:
    df_d = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    df_d = df_d[df_d["event_type"] == CALL_TYPE]
    parts.append(df_d)
calls = pd.concat(parts, ignore_index=True)

calls = detect_alarm_scales(calls)
# calls now has:
#   ici_s, bout_id, bout_size, bout_position, bout_kind  (finest scale)
#   event_id, event_size, event_position                 (coarser; no _kind on purpose)

# Per-bout / per-event summary tables. extra_first_cols copies the alarm
# classifier-confidence column from each row's first call. finer_scale="bout"
# adds an n_bouts column to events_meta.
bouts_meta  = summarize_scale(
    calls, prefix="bout",
    extra_first_cols=("meanprob_alarm",),
)
events_meta = summarize_scale(
    calls, prefix="event",
    extra_first_cols=("meanprob_alarm",),
    finer_scale="bout",
)

print(f"{len(calls):,} {CALL_TYPE} calls total "
      f"-> {len(bouts_meta):,} bouts, {len(events_meta):,} events")

# --- Bout-scale summaries ---
print("\nBouts per (date, bout_kind):")
print(bouts_meta.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))

# Size stats specifically for "real" multi-call bouts (kind == "in_bout").
# Singletons dominate the raw distribution and would pull medians down to 1.
in_bout_only = bouts_meta[bouts_meta["bout_kind"] == "in_bout"]
print(f"\nIn-bout-only size stats per date "
      f"(bouts with n_calls >= {BOUT_THRESHOLDS[CALL_TYPE]['min_bout_size']}):")
print(in_bout_only.groupby("date_folder")["n_calls"].agg(["count", "mean", "median", "max"]))

# --- Event-scale summaries ---
print("\nEvents per date (n_bouts and n_calls summaries):")
print(
    events_meta.groupby("date_folder").agg(
        n_events=("n_calls", "size"),
        median_n_bouts=("n_bouts", "median"),
        max_n_bouts=("n_bouts", "max"),
        median_n_calls=("n_calls", "median"),
        max_n_calls=("n_calls", "max"),
    )
)

## Inter-call gap (ICG) per date

Silent gap between consecutive alarm calls (`this.start − previous.stop`), one row per panel for each date. Log-scale x because ICGs span ms (within bouts) to hours (between events).

Dotted lines mark the two thresholds from `ALARM_SCALES` (which act on the gap): gaps below 2 s stay within a bout; 2–30 s gaps stay within an event but start a new bout; gaps above 30 s start a new event.

(If you instead want **start-to-start intervals** — period, `1/median = Hz` — use `calls["ici_s"]`.)

In [ ]:
# All inter-call GAPS (calls["icg_s"]) per date.
#
# ICG = this.start - previous.stop (silent gap). We plot the GAP because
# the bout/event thresholds are defined on the gap, so the threshold lines
# are directly interpretable here. If you want start-to-start INTERVALS
# (period; 1/median = Hz), use calls["ici_s"] instead.
#
# We plot the histogram of log10(ICG). The convention for distributions
# that span many decades (here: tens of ms to hours). Density is then w.r.t.
# log10(x), so equal visible area = equal proportion of calls in that decade
# range.

MIN_PLOT_S = 0.03    # 30 ms - drop sub-30ms gaps (segmentation artifacts)

icg_s = calls["icg_s"].dropna()
n_total = len(icg_s)
icg_s = icg_s[icg_s >= MIN_PLOT_S]
print(f"ICG plot: showing {len(icg_s):,}/{n_total:,} gaps "
      f"(dropped {n_total - len(icg_s):,} gaps < {MIN_PLOT_S*1000:.0f} ms)")

# Log10-transform once, then use LINEAR bins on the transformed data.
log_icg = np.log10(icg_s)
bins = np.linspace(log_icg.min(), log_icg.max(), 60)

# Thresholds (both defined on the GAP - that's why we plot the gap here).
bout_thr  = ALARM_SCALES[0]["max_icg_s"]
event_thr = ALARM_SCALES[1]["max_icg_s"]

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.2 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = calls[calls["date_folder"] == date]["icg_s"].dropna()
    sub = sub[sub >= MIN_PLOT_S]
    ax.hist(np.log10(sub), bins=bins, density=True, color="gray", edgecolor="white")
    ax.axvline(np.log10(bout_thr),  color="red",  linestyle=":", label=f"bout thr ({bout_thr}s)")
    ax.axvline(np.log10(event_thr), color="blue", linestyle=":", label=f"event thr ({event_thr}s)")
    ax.set_title(f"{date}  (n={len(sub):,})", loc="left")
    ax.set_ylabel("density (per log10 s)")
    ax.legend(loc="upper right", fontsize=8)

tick_seconds = [0.05, 0.1, 0.5, 1, 2, 5, 10, 30, 60, 300, 600, 3600, 36000]
tick_labels  = ["50ms", "100ms", "500ms", "1s", "2s", "5s", "10s", "30s",
                "1m", "5m", "10m", "1h", "10h"]
axes[-1].set_xticks(np.log10(tick_seconds))
axes[-1].set_xticklabels(tick_labels, fontsize=8)
axes[-1].set_xlabel("Inter-call gap (this.start - previous.stop)")

fig.suptitle("Inter-call gaps per date (log10-binned)")
fig.tight_layout()
save_fig(fig, "alarm_icg_per_date")
plt.show()

In [ ]:
# Within-bout calling rate per date - LINEAR x-axis version.
MIN_SIZE = 5
CV_MIN_CALLS = 10   # min calls per bout for within-bout cv_ici to be meaningful

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE].copy()
pool["rate_hz"] = (pool["bout_size"] - 1) / pool["duration_s"]
pool = pool[pool["rate_hz"] > 0]

# Cap the x-axis at the 99th percentile so the right-skew tail doesn't squash
# the bulk of the distribution.
rate_cap = np.percentile(pool["rate_hz"].values, 99)
n_above  = int((pool["rate_hz"] > rate_cap).sum())
print(f"Linear-axis rate plot: capping x at 99th pct = {rate_cap:.2f} Hz "
      f"({n_above} bouts above the cap not shown)")

bins = np.linspace(0, rate_cap, 40)

fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(3.0 * len(DATES_TO_PLOT), 3.5),
                         sharex=True, sharey=True)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = pool[pool["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue
    rate = sub["rate_hz"].values
    ax.hist(rate[rate <= rate_cap], bins=bins, density=True,
            color="gray", edgecolor="white", alpha=0.7)
    med  = np.median(rate)
    mean = np.mean(rate)
    # Within-bout CV (cv_ici): median across bouts with enough calls.
    cv_sub = sub.loc[sub["bout_size"] >= CV_MIN_CALLS, "cv_ici"].dropna()
    cv_med = cv_sub.median() if len(cv_sub) else np.nan
    ax.axvline(med, color="red", linewidth=1.2, linestyle="--",
               label=(f"median = {med:.2f} Hz\n"
                      f"within-bout CV = {cv_med:.2f}"))
    ax.set_title(f"{date}  (n={len(sub):,} bouts)", fontsize=10)
    ax.set_xlabel("within-bout calling rate (Hz)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(0, 3)
    ax.set_xticks([0, 1, 2, 3]) 
    ax.spines[["top", "right"]].set_visible(False)
 

axes[0].set_ylabel("density")
fig.suptitle(f"Within-bout calling rate per date (size >= {MIN_SIZE}, linear x)",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "alarm_within_bout_rate_per_date_linear")
plt.show()


### Pooled across all dates

Per-bout calling rate (`(size − 1) / duration_s`), pooled across all dates. Linear x-axis, capped at 3 Hz. Within-bout CV (`cv_ici`, median across bouts with ≥10 calls) shown in the legend.

In [ ]:
# Within-bout calling rate pooled across all dates (linear x) - POSTER VERSION.
MIN_SIZE     = 5
CV_MIN_CALLS = 10
FS = 24   # poster font size

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE].copy()
pool["rate_hz"] = (pool["bout_size"] - 1) / pool["duration_s"]
pool = pool[pool["rate_hz"] > 0]
rates = pool["rate_hz"].values

med = np.median(rates)
cv_sub = pool.loc[pool["bout_size"] >= CV_MIN_CALLS, "cv_ici"].dropna()
cv_med = cv_sub.median() if len(cv_sub) else np.nan
print(f"Pooled rate plot: {len(rates):,} bouts. median={med:.2f} Hz, "
      f"within-bout CV={cv_med:.2f}")

X_MIN, X_MAX = 0, 3

fig, ax = plt.subplots(figsize=(10, 7))
bins = np.linspace(X_MIN, X_MAX, 40)
in_range = rates[(rates >= X_MIN) & (rates <= X_MAX)]
n_above  = int((rates > X_MAX).sum())
ax.hist(in_range, bins=bins, density=True,
        color="gray", edgecolor="white", alpha=0.7)

ax.axvline(med, color="red", linewidth=3, linestyle="--",
           label=f"median = {med:.2f} Hz")

ax.set_xlim(X_MIN, X_MAX)
ax.set_xticks([0, 1, 2, 3])
ax.set_xlabel("Within-bout calling rate (Hz)", fontsize=FS)
ax.set_ylabel("Density", fontsize=FS)
ax.set_yticks([0, 1.5])
ax.tick_params(axis="both", which="major", labelsize=FS, width=1.5, length=6)
ax.set_title(f"Within-bout calling rate (n={len(rates):,})", fontsize=FS)
ax.legend(fontsize=FS, loc="upper right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(1.5)

fig.tight_layout()
save_fig(fig, "alarm_within_bout_rate_pooled")
plt.show()

## Inter-bout gap per date

One scale up. `bouts_meta["gap_s"]` is the silent gap between consecutive bouts (`this_bout.start − previous_bout.stop`). By construction every value is ≥ 2 s (anything shorter would have been *within* the same bout).

The red dotted line marks the bout threshold (left edge — gaps can't be smaller). The blue dotted line marks the event threshold (30 s): gaps below it stay inside one event, gaps above it cross between events.

(For start-to-start **inter-bout intervals**, use `bouts_meta["interval_s"]`.)

In [ ]:
# Inter-bout gaps (gap_s on bouts_meta).
# gap_s = this_bout.start - previous_bout.stop. All values are >= bout
# threshold (2 s) by construction. The event threshold (30 s) separates
# gaps that stay within one event vs cross between events.
#
# Same log10-transform approach as the ICG plot: histogram of log10(gap_s)
# with linear bins. Density is then w.r.t. log10(x), so heavy tails stay visible.

ibg_s = bouts_meta["gap_s"].dropna()
ibg_s = ibg_s[ibg_s > 0]

log_ibg = np.log10(ibg_s)
bins = np.linspace(log_ibg.min(), log_ibg.max(), 60)

bout_thr  = ALARM_SCALES[0]["max_icg_s"]
event_thr = ALARM_SCALES[1]["max_icg_s"]

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.2 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = bouts_meta[bouts_meta["date_folder"] == date]["gap_s"].dropna()
    sub = sub[sub > 0]
    ax.hist(np.log10(sub), bins=bins, density=True, color="gray", edgecolor="white")
    ax.axvline(np.log10(bout_thr),  color="red",  linestyle=":", label=f"bout thr ({bout_thr}s)")
    ax.axvline(np.log10(event_thr), color="blue", linestyle=":", label=f"event thr ({event_thr}s)")
    ax.set_title(f"{date}  (n={len(sub):,})", loc="left")
    ax.set_ylabel("density (per log10 s)")
    ax.legend(loc="upper right", fontsize=8)

tick_seconds = [2, 5, 10, 30, 60, 300, 600, 3600, 36000, 86400]
tick_labels  = ["2s", "5s", "10s", "30s", "1m", "5m", "10m", "1h", "10h", "1d"]
axes[-1].set_xticks(np.log10(tick_seconds))
axes[-1].set_xticklabels(tick_labels, fontsize=8)
axes[-1].set_xlabel("Inter-bout gap")

fig.suptitle("Inter-bout gaps per date (log10-binned)")
fig.tight_layout()
save_fig(fig, "alarm_inter_bout_gap_per_date")
plt.show()

## Inter-event gap per date

Same idea, one scale coarser. Each row in `events_meta` carries `gap_s` = gap from the previous event's last call to this event's first call. All values are ≥ 30 s by construction.

(For start-to-start **inter-event intervals**, use `events_meta["interval_s"]`.)

In [ ]:
# Inter-event gaps (gap_s on events_meta).
# gap_s = this_event.start - previous_event.stop. All values are >= event
# threshold (30 s) by construction.
#
# Same log10-transform approach as the ICG / IBG plots.

ieg_s = events_meta["gap_s"].dropna()
ieg_s = ieg_s[ieg_s > 0]

log_ieg = np.log10(ieg_s)
bins = np.linspace(log_ieg.min(), log_ieg.max(), 60)

event_thr = ALARM_SCALES[1]["max_icg_s"]

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.2 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = events_meta[events_meta["date_folder"] == date]["gap_s"].dropna()
    sub = sub[sub > 0]
    ax.hist(np.log10(sub), bins=bins, density=True, color="gray", edgecolor="white")
    ax.axvline(np.log10(event_thr), color="blue", linestyle=":", label=f"event thr ({event_thr}s)")
    ax.set_title(f"{date}  (n={len(sub):,})", loc="left")
    ax.set_ylabel("density (per log10 s)")
    ax.legend(loc="upper right", fontsize=8)

tick_seconds = [30, 60, 300, 600, 3600, 36000, 86400]
tick_labels  = ["30s", "1m", "5m", "10m", "1h", "10h", "1d"]
axes[-1].set_xticks(np.log10(tick_seconds))
axes[-1].set_xticklabels(tick_labels, fontsize=8)
axes[-1].set_xlabel("Inter-event gap")

fig.suptitle("Inter-event gaps per date (log10-binned)")
fig.tight_layout()
save_fig(fig, "alarm_inter_event_gap_per_date")
plt.show()

## Calls per hour, per date

Hourly rate of alarm calls across the day, averaged across all observed days, with the dark cycle shaded gray. One panel per date.

In [ ]:
# Average alarm calls per hour-of-day, per date.
# Recipe (also used for bouts and events below):
#   1. For each date, get each row's hour-of-day from its timestamp.
#   2. Count rows per hour. Reindex to range(24) so empty hours show as 0.
#   3. Divide by the number of distinct days observed → average per day per hour.
#   4. Shade the dark-cycle hours (light hours stay white).

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.0 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = calls[calls["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue

    hours  = sub["start_time_real"].dt.hour
    n_days = sub["start_time_real"].dt.normalize().nunique()
    counts = hours.value_counts().reindex(range(24), fill_value=0)
    rate   = counts / n_days   # mean count per hour per day

    ax.bar(rate.index, rate.values, color="gray", edgecolor="white", linewidth=0.5)

    # Shade DARK hours (light hours stay white).
    light_start, light_end = LIGHT_OVERRIDES.get(date, LIGHT_DEFAULT)
    if light_start < light_end:
        # Normal case: light is a continuous window inside the day.
        ax.axvspan(0, light_start, color="lightgray", alpha=0.3, zorder=0)
        ax.axvspan(light_end, 24,  color="lightgray", alpha=0.3, zorder=0)
    else:
        # Light window wraps midnight (start > end). Shade the single dark block in between.
        ax.axvspan(light_end, light_start, color="lightgray", alpha=0.3, zorder=0)

    ax.set_title(f"{date}  ({len(sub):,} calls over {n_days} days)", loc="left")
    ax.set_ylabel("Calls per hour\n(per day average)")

ticks = list(range(0, 25, 2))
axes[-1].set_xticks(ticks)
axes[-1].set_xticklabels([f"{h:02d}:00" for h in ticks])
axes[-1].set_xlabel("Hour of day")
axes[-1].set_xlim(0, 23)

fig.suptitle("Alarm calls per hour, per date")
fig.tight_layout()
save_fig(fig, "alarm_calls_per_hour_per_date")
plt.show()

## Bouts per hour, per date

Same shape, but counting **bouts** (one row per bout in `bouts_meta`) — each bar is the average number of bouts that *started* in that hour, across days.

In [ ]:
# Average bouts per hour-of-day, per date.
fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.0 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = bouts_meta[bouts_meta["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue

    hours  = sub["start_time"].dt.hour
    n_days = sub["start_time"].dt.normalize().nunique()
    counts = hours.value_counts().reindex(range(24), fill_value=0)
    rate   = counts / n_days

    ax.bar(rate.index, rate.values, color="gray", edgecolor="white", linewidth=0.5)

    # Shade DARK hours (light hours stay white).
    light_start, light_end = LIGHT_OVERRIDES.get(date, LIGHT_DEFAULT)
    if light_start < light_end:
        ax.axvspan(0, light_start, color="lightgray", alpha=0.3, zorder=0)
        ax.axvspan(light_end, 24,  color="lightgray", alpha=0.3, zorder=0)
    else:
        ax.axvspan(light_end, light_start, color="lightgray", alpha=0.3, zorder=0)

    ax.set_title(f"{date}  ({len(sub):,} bouts over {n_days} days)", loc="left")
    ax.set_ylabel("Bouts per hour\n(per day average)")

ticks = list(range(0, 25, 2))
axes[-1].set_xticks(ticks)
axes[-1].set_xticklabels([f"{h:02d}:00" for h in ticks])
axes[-1].set_xlabel("Hour of day")
axes[-1].set_xlim(0, 23)

fig.suptitle("Alarm bouts per hour, per date")
fig.tight_layout()
save_fig(fig, "alarm_bouts_per_hour_per_date")
plt.show()

## Events per hour, per date

One row per event in `events_meta`. Each bar = events that started in that hour, averaged across days. Dark hours shaded.

In [ ]:
# Average events per hour-of-day, per date.
fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.0 * len(DATES_TO_PLOT)),
    sharex=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = events_meta[events_meta["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue

    hours  = sub["start_time"].dt.hour
    n_days = sub["start_time"].dt.normalize().nunique()
    counts = hours.value_counts().reindex(range(24), fill_value=0)
    rate   = counts / n_days

    ax.bar(rate.index, rate.values, color="gray", edgecolor="white", linewidth=0.5)

    # Shade DARK hours (light hours stay white).
    light_start, light_end = LIGHT_OVERRIDES.get(date, LIGHT_DEFAULT)
    if light_start < light_end:
        ax.axvspan(0, light_start, color="lightgray", alpha=0.3, zorder=0)
        ax.axvspan(light_end, 24,  color="lightgray", alpha=0.3, zorder=0)
    else:
        ax.axvspan(light_end, light_start, color="lightgray", alpha=0.3, zorder=0)

    ax.set_title(f"{date}  ({len(sub):,} events over {n_days} days)", loc="left")
    ax.set_ylabel("Events per hour\n(per day average)")

ticks = list(range(0, 25, 2))
axes[-1].set_xticks(ticks)
axes[-1].set_xticklabels([f"{h:02d}:00" for h in ticks])
axes[-1].set_xlabel("Hour of day")
axes[-1].set_xlim(0, 23)

fig.suptitle("Alarm events per hour, per date")
fig.tight_layout()
save_fig(fig, "alarm_events_per_hour_per_date")
plt.show()

## CV of within-bout ICI per date

**CV** (coefficient of variation) = `std(ICI) / mean(ICI)`, computed per bout. Already stored in `bouts_meta["cv_ici"]` (start-to-start period-based). Measures within-bout rhythm regularity:

- `CV ≈ 0` → perfectly regular (every ICI the same)
- `CV = 1` → Poisson-like (exponentially distributed ICIs)
- `CV > 1` → more bursty than Poisson

Restrict to bouts with at least `MIN_CALLS_FOR_CV` calls so each per-bout CV is computed from enough samples to be meaningful.

(If you want the old `alarm_bouts_3` numbers exactly, use `cv_icg` instead — that's the gap-based version, which was what their `cv_ici` actually computed under the old naming.)

In [ ]:
# CV of within-bout ICI per family - POSTER VERSION.
# CV = std(ICI) / mean(ICI), computed per bout - already on bouts_meta as cv_ici.
# Reference line at CV=1 = Poisson-like irregularity.
#
# Each date_folder corresponds to a different gerbil family; relabel as
# "Family 1, 2, ..." in the order of DATES_TO_PLOT.
MIN_CALLS_FOR_CV = 10
FS = 24   # poster font size

bins = np.linspace(0, 1.5, 40)

fig, ax = plt.subplots(figsize=(11, 7))

# One color per family (matplotlib's default cycle).
date_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"][:len(DATES_TO_PLOT)]

for i, (date, color) in enumerate(zip(DATES_TO_PLOT, date_colors), start=1):
    sub = bouts_meta[
        (bouts_meta["date_folder"] == date)
        & (bouts_meta["bout_size"] >= MIN_CALLS_FOR_CV)
    ]["cv_ici"].dropna()
    if sub.empty:
        continue
    ax.hist(
        sub, bins=bins, histtype="step", linewidth=3, density=True, color=color,
        label=f"Family {i}  (n={len(sub):,}, med={sub.median():.2f})",
    )

ax.axvline(1.0, color="black", linestyle="--", linewidth=2, label="Poisson (CV=1)")
ax.set_xlabel("CV of within-bout ICI  (std / mean)", fontsize=FS)
ax.set_ylabel("Density", fontsize=FS)
ax.tick_params(axis="both", which="major", labelsize=FS, width=1.5, length=6)
ax.set_title(f"Within-bout regularity  (bouts ≥ {MIN_CALLS_FOR_CV} calls)", fontsize=FS)
ax.legend(fontsize=FS, loc="upper right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(1.5)

fig.tight_layout()
save_fig(fig, f"alarm_within_bout_cv_ici_per_family_min{MIN_CALLS_FOR_CV}")
plt.show()

## Bout size distribution per date

How many calls are in each bout? Singletons excluded (size >= 2). Bout sizes are heavy-tailed, so we plot `log10(size)` with linspace bins + `density=True` (per the project's log-bins convention).

In [ ]:
# Bout-size distribution per date (singletons excluded).
MIN_SIZE = 5

fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(3.0 * len(DATES_TO_PLOT), 3.5),
                         sharex=True, sharey=True)

# Determine global log-size range so bins are consistent across panels.
pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE]
log_sizes_all = np.log10(pool["bout_size"].values)
bins = np.linspace(log_sizes_all.min(), log_sizes_all.max() + 0.01, 30)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = pool[pool["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off(); continue
    log_size = np.log10(sub["bout_size"].values)
    ax.hist(log_size, bins=bins, density=True,
            color="gray", edgecolor="white", alpha=0.7)
    ax.set_title(f"{date}  (n={len(sub):,} bouts)", fontsize=10)
    ax.set_xlabel("# calls in bout")
    ax.axvline(np.log10(np.median(sub["bout_size"].values)),
               color="red", linewidth=1.2, linestyle="--",
               label=f"median = {int(np.median(sub['bout_size'])):d}")
    ax.legend(fontsize=8, loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)

# Friendly tick labels at meaningful counts.
tick_positions = [2, 3, 5, 10, 20, 50, 100]
tick_log = [np.log10(t) for t in tick_positions]
axes[0].set_xticks(tick_log)
axes[0].set_xticklabels([str(t) for t in tick_positions])
axes[0].set_ylabel("density")

fig.suptitle(f"Bout size distribution (size >= {MIN_SIZE})", y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "alarm_bout_size_distribution")
plt.show()


### Same, linear x-axis

Bout-size distribution with a **linear** x-axis (integer bins, one per call count) instead of log10. Bout size is heavily right-skewed, so the x-axis is capped at the 99th percentile (a few large bouts beyond that are dropped — count printed).

Compare against the log version above: log shows the full heavy tail; linear is more intuitive for the common small-bout range.

In [ ]:
# Bout-size distribution per date - LINEAR x-axis version.
MIN_SIZE = 5

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE]

# Cap at 99th percentile so the heavy tail doesn't squash the bulk.
size_cap = int(np.percentile(pool["bout_size"].values, 99))
n_above  = int((pool["bout_size"] > size_cap).sum())
print(f"Linear bout-size plot: capping x at 99th pct = {size_cap} calls "
      f"({n_above} bouts above the cap not shown)")

# Integer-centered bins (each bin = one call-count value).
bins = np.arange(MIN_SIZE - 0.5, size_cap + 1.5, 1.0)

fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(3.0 * len(DATES_TO_PLOT), 3.5),
                         sharex=True, sharey=True)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = pool[pool["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue
    sizes = sub["bout_size"].values
    ax.hist(sizes[sizes <= size_cap], bins=bins, density=True,
            color="steelblue", edgecolor="white", alpha=0.7)
    med = np.median(sizes)
    ax.axvline(med, color="red", linewidth=1.2, linestyle="--",
               label=f"median = {int(med)} calls")
    ax.set_title(f"{date}  (n={len(sub):,} bouts)", fontsize=10)
    ax.set_xlabel("# calls in bout")
    ax.legend(fontsize=8, loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("density")
fig.suptitle(f"Bout size distribution per date (size >= {MIN_SIZE}, linear x)",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "alarm_bout_size_distribution_linear")
plt.show()


### Pooled across all dates

Bout-size distribution pooled across all dates. Log x-axis (heavy tail visible), x clipped to 5-300 calls.

In [ ]:
# Bout-size distribution pooled across all dates (log x) - POSTER VERSION.
MIN_SIZE = 5
FS = 24   # poster font size

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE]
sizes = pool["bout_size"].values
print(f"Pooled bout-size plot: {len(sizes):,} bouts (all dates, size >= {MIN_SIZE})")

X_MIN, X_MAX = 5, 300

fig, ax = plt.subplots(figsize=(10, 7))
log_sizes = np.log10(sizes)
bins = np.linspace(np.log10(X_MIN), np.log10(X_MAX), 35)
ax.hist(log_sizes, bins=bins, density=True,
        color="gray", edgecolor="white", alpha=0.7)

med = int(np.median(sizes))
ax.axvline(np.log10(med), color="red", linewidth=3, linestyle="--",
           label=f"median = {med} calls")

tick_positions = [5, 10, 50, 100, 300]
ax.set_xticks([np.log10(t) for t in tick_positions])
ax.set_xticklabels([str(t) for t in tick_positions])
ax.set_xlim(np.log10(X_MIN), np.log10(X_MAX))
ax.set_xlabel("No. calls in bout", fontsize=FS)
ax.set_ylabel("Density", fontsize=FS)
ax.set_yticks([0, 3])
ax.tick_params(axis="both", which="major", labelsize=FS, width=1.5, length=6)

ax.set_title(f"Bout size distribution (n={len(sizes):,})", fontsize=FS)
ax.legend(fontsize=FS, loc="upper right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_linewidth(1.5)

fig.tight_layout()
save_fig(fig, "alarm_bout_size_distribution_pooled")
plt.show()

## Bout size vs bout duration

One point per bout (singletons excluded). x = number of calls, y = bout duration in seconds. Both axes log because both are heavy-tailed.

If calls were emitted at a constant rate, the cloud would fall along a single line with slope 1 on the log-log plot (`duration ≈ const · n_calls`). Deviations from that line tell you about rate variability across bouts.

In [ ]:
# Scatter: bout size vs bout duration, per date.
MIN_SIZE = 2

fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(3.0 * len(DATES_TO_PLOT), 3.5),
                         sharex=True, sharey=True)

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE]

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = pool[pool["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off(); continue
    ax.scatter(
        sub["bout_size"], sub["duration_s"],
        s=8, alpha=0.25, color="steelblue", edgecolor="none",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("# calls in bout")
    ax.set_title(f"{date}  (n={len(sub):,} bouts)", fontsize=10)

axes[0].set_ylabel("bout duration (s)")
fig.suptitle("Bout size vs duration (one dot = one bout, singletons excluded)",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "alarm_bout_size_vs_duration")
plt.show()


## Within-bout calling rate vs bout size

Test directly whether calling rate depends on bout size. For each date, bin bouts by size on a log scale, then plot the **median rate per bin** with an **IQR band**. Faint scatter of individual bouts behind.

If rate is constant across bout sizes, the median line is flat. If long bouts call faster/slower, you'll see a clear trend.

In [ ]:
# Within-bout calling rate vs bout size, per date.
# Per-bin median + IQR band; scatter cloud of all bouts behind.

MIN_SIZE  = 2

pool = bouts_meta[bouts_meta["bout_size"] >= MIN_SIZE].copy()
pool["rate_hz"] = (pool["bout_size"] - 1) / pool["duration_s"]
pool = pool[pool["rate_hz"] > 0]

# Log-spaced size bins.
size_edges  = np.array([2, 3, 4, 6, 10, 16, 25, 40, 64, 100, 160, 250])
bin_centers = np.sqrt(size_edges[:-1] * size_edges[1:])   # geometric mid

fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(3.0 * len(DATES_TO_PLOT), 3.5),
                         sharex=True, sharey=True)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = pool[pool["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off(); continue

    # Faint scatter cloud.
    ax.scatter(sub["bout_size"], sub["rate_hz"],
               s=8, alpha=0.15, color="steelblue", edgecolor="none", zorder=2)

    # Per-bin median + IQR.
    sub_binned = sub.copy()
    sub_binned["bin_idx"] = np.digitize(sub_binned["bout_size"].values, size_edges) - 1
    sub_binned = sub_binned[(sub_binned["bin_idx"] >= 0)
                              & (sub_binned["bin_idx"] < len(size_edges) - 1)]
    by_bin = sub_binned.groupby("bin_idx")["rate_hz"]
    med = by_bin.median()
    q25 = by_bin.quantile(0.25)
    q75 = by_bin.quantile(0.75)
    n_per_bin = by_bin.size()

    # Drop bins with too few bouts to be meaningful.
    valid = n_per_bin >= 5
    xs = bin_centers[med.index][valid.values]
    ax.fill_between(xs, q25[valid].values, q75[valid].values,
                    color="navy", alpha=0.20, zorder=3, label="IQR")
    ax.plot(xs, med[valid].values, "-o",
            color="navy", linewidth=1.7, markersize=5, zorder=4, label="median")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("# calls in bout")
    ax.set_title(f"{date}  (n={len(sub):,} bouts)", fontsize=10)

axes[0].set_ylabel("within-bout calling rate (Hz)")
axes[0].legend(fontsize=8, loc="lower right")

fig.suptitle("Within-bout calling rate vs bout size",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, "alarm_bout_rate_vs_size")
plt.show()


### Inspect fast-rate bouts (>=5 Hz)

The edges of the rate-vs-size cloud show bouts calling at 5-10 Hz. Sample 30, plot each as a spectrogram of its full duration, with white vertical lines marking individual call boundaries. Goal: see whether these are real fast-calling sequences or segmentation artifacts (e.g., one call split into multiple).

In [ ]:
# Spectrogram inspection of fast-rate alarm bouts.
import librosa
from vocalization_analysis.acoustic_features import load_call_slice

if HOST != "Linux":
    raise RuntimeError("Spectrograms need raw WAVs - run on the cluster.")

RATE_MIN_HZ = 5.0
MIN_SIZE    = 3
N_TO_SHOW   = 30
N_COLS      = 5
PADDING_SEC = 0.05    # padding before bout-start and after bout-stop
NFFT_PLOT   = 512
HOP_PLOT    = 128

# Find fast bouts.
bouts_with_rate = bouts_meta.copy()
bouts_with_rate["rate_hz"] = (bouts_with_rate["bout_size"] - 1) / bouts_with_rate["duration_s"]
fast = bouts_with_rate[
    (bouts_with_rate["rate_hz"] >= RATE_MIN_HZ)
    & (bouts_with_rate["bout_size"] >= MIN_SIZE)
]
print(f"Fast bouts (rate >= {RATE_MIN_HZ} Hz, size >= {MIN_SIZE}): {len(fast)}")
print(f"Per date: \n{fast.groupby('date_folder').size().to_string()}")

n_take  = min(N_TO_SHOW, len(fast))
sample  = fast.sample(n_take, random_state=0)
n_rows  = (n_take + N_COLS - 1) // N_COLS

fig, axes = plt.subplots(n_rows, N_COLS,
                         figsize=(3.2 * N_COLS, 2.2 * n_rows),
                         squeeze=False)

for i in range(n_rows * N_COLS):
    r, c = divmod(i, N_COLS)
    ax = axes[r, c]
    if i >= n_take:
        ax.set_axis_off(); continue

    bout_id  = sample.index[i]
    bout_row = sample.iloc[i]
    bout_calls = calls[calls["bout_id"] == bout_id].sort_values("start_time_file_sec")
    if bout_calls.empty:
        ax.set_axis_off(); continue

    # Cross-file or cross-channel bouts: skip (rare).
    if bout_calls["file_num"].nunique() != 1 or bout_calls["channel"].nunique() != 1:
        ax.text(0.5, 0.5, "cross-file bout", transform=ax.transAxes,
                ha="center", va="center", fontsize=8)
        ax.set_axis_off(); continue

    first_call = bout_calls.iloc[0]
    last_call  = bout_calls.iloc[-1]

    win_start = max(0.0, first_call["start_time_file_sec"] - PADDING_SEC)
    win_stop  = last_call["stop_time_file_sec"] + PADDING_SEC

    try:
        y, sr = load_call_slice(
            BASE_PROCESSED_AUDIO,
            first_call["date_folder"], first_call["exp"],
            first_call["channel"],     first_call["file_num"],
            win_start, win_stop, pad_sec=0.0,
        )
    except Exception:
        ax.set_axis_off(); continue
    if len(y) < NFFT_PLOT:
        ax.set_axis_off(); continue

    actual_dur      = len(y) / sr
    win_stop_actual = win_start + actual_dur

    S    = np.abs(librosa.stft(y.astype(np.float32),
                                n_fft=NFFT_PLOT, hop_length=HOP_PLOT, window="hann"))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_PLOT)
    ax.imshow(S_db, aspect="auto", origin="lower", cmap="magma", vmin=-60, vmax=0,
              extent=[win_start, win_stop_actual, freqs[0] / 1000, freqs[-1] / 1000])
    ax.set_ylim(1, 60)

    # Overlay every call's start/stop within the bout.
    for _, cr in bout_calls.iterrows():
        ax.axvline(cr["start_time_file_sec"], color="white",
                   linewidth=1.1, alpha=0.8)
        ax.axvline(cr["stop_time_file_sec"],  color="white",
                   linewidth=1.1, alpha=0.6, linestyle="--")

    # Number badge.
    ax.text(0.03, 0.95, f"{i+1}",
            transform=ax.transAxes, fontsize=18, fontweight="bold",
            color="white", va="top", ha="left",
            bbox=dict(facecolor="black", alpha=0.6, pad=2, edgecolor="none"))

    ax.set_title(
        f"{first_call['date_folder']}  exp{int(first_call['exp'])}/"
        f"f{int(first_call['file_num']):03d}  ·  "
        f"n={int(bout_row['bout_size'])}, {bout_row['rate_hz']:.1f}Hz",
        fontsize=8,
    )

fig.suptitle(f"Fast-rate alarm bouts (rate >= {RATE_MIN_HZ} Hz)  ·  "
             f"white lines = call boundaries (solid=start, dashed=stop)",
             y=1.01, fontsize=11)
fig.tight_layout()
save_fig(fig, "alarm_fast_rate_bouts_examples")
plt.show()


## Foot-drumming detector + continuation-call detector

Per-bout, three stacked panels:

1. **Top — autocorrelation** of the **low-band (`LOW_BAND_HZ`)** envelope, x-axis now extends to 2 s so slow drums (~1 Hz) are visible. Green band = drum-rate range, red = best fundamental, orange dashed = harmonics.
2. **Middle — spectrogram in the alarm band** (`HIGH_BAND_HZ`, default `(20 kHz, 50 kHz)`). Spectrogram y-axis follows whatever you set HIGH_BAND_HZ to. Cyan curve = high-band envelope; cyan vertical ticks = detected high-band events.
3. **Bottom — spectrogram in the drum band** (`LOW_BAND_HZ`).

Title now shows `drum=… calls=N/drums=M (ratio=N/M) [class] rate=…Hz`.

**Classification by ratio of calls to drum impulses** (more robust than a fixed call-count threshold):

| Class | Condition |
|---|---|
| `drum_only` (green) | `drum_score ≥ 0.3` AND `(n_calls / n_drums) < 0.5` |
| `drum+calls` (orange) | `drum_score ≥ 0.3` AND `n_calls ≥ 2` AND `(n_calls / n_drums) ≥ 0.5` |
| `calls_only` (red) | `drum_score < 0.3` AND `n_calls ≥ 2` |
| `quiet` (gray) | otherwise |

So a bout with 9 detected drums + 3 calls is `drum_only` (ratio 0.33), but 9 drums + 6 calls would be `drum+calls` (ratio 0.67).

In [ ]:
# Combined drum-tail + continuation-call detector, per bout.
import gc
import librosa
from scipy.signal import find_peaks
from vocalization_analysis.acoustic_features import load_call_slice

if HOST != "Linux":
    raise RuntimeError("Audio loading needed - run on the cluster.")

# === Config ===
POST_WIN_SEC      = 5.0
MIN_BOUT_SIZE     = 5
N_TO_SHOW         = 18
N_COLS            = 3
LOW_BAND_HZ       = (0, 3000)
HIGH_BAND_HZ      = (20000, 50000)   # for continuation-call detection
ENV_HOP_MS        = 5
DRUM_LAG_RANGE_S  = (0.050, 2.000)   # 0.5-20 Hz - cover both 'fast' (5-15 Hz) and 'slow' (~1 Hz) drum trains
NFFT_DRUM         = 1024
NFFT_SPEC         = 512
HOP_SPEC          = 256              # ~2 ms per frame in the displayed spec
HIGH_EVENT_PROM   = 0.3              # min normalized-envelope prominence
HIGH_EVENT_MIN_S  = 0.050            # min separation between high-band events

# Class thresholds.
DRUM_SCORE_T = 0.3
CALL_COUNT_T = 2

# Class thresholds: drum_only requires fewer than CALL_RATIO_MAX calls per drum event.
CALL_RATIO_MAX = 0.5

def classify(drum_score, n_calls, n_drum_events):
    drum_present = drum_score >= DRUM_SCORE_T
    # Ratio of detected calls to detected drum impulses.
    ratio = n_calls / max(n_drum_events, 1)
    calls_dominant = (n_calls >= CALL_COUNT_T) and (ratio >= CALL_RATIO_MAX)
    if drum_present and not calls_dominant: return "drum_only",  "#1f9d55"
    if drum_present and calls_dominant:     return "drum+calls", "#d68910"
    if (not drum_present) and (n_calls >= CALL_COUNT_T): return "calls_only", "#c0392b"
    return "quiet", "#808080"

def detect_drum_tail_v2(y, sr):
    if len(y) < NFFT_DRUM:
        return None
    hop = int(round(ENV_HOP_MS * sr / 1000))
    S = np.abs(librosa.stft(y.astype(np.float32),
                             n_fft=NFFT_DRUM, hop_length=hop, window="hann"))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_DRUM)
    in_band = (freqs >= LOW_BAND_HZ[0]) & (freqs <= LOW_BAND_HZ[1])
    env = S[in_band].sum(axis=0)
    env_c = env - env.mean()
    n = len(env_c)
    if n < 10:
        return None
    ac = np.correlate(env_c, env_c, mode="full")[n-1:]
    if ac[0] <= 0:
        return None
    ac = ac / ac[0]
    ac_t = np.arange(len(ac)) * (hop / sr)

    mask = (ac_t >= DRUM_LAG_RANGE_S[0]) & (ac_t <= DRUM_LAG_RANGE_S[1])
    if not mask.any():
        return None
    lags_in, ac_in = ac_t[mask], ac[mask]
    peaks, props = find_peaks(ac_in, prominence=0.0)
    if len(peaks) == 0:
        return {"autocorr": ac, "autocorr_t": ac_t,
                "drum_prom": 0.0, "drum_lag": np.nan,
                "harmonic_lags": [], "harmonic_proms": [],
                "n_harmonics": 0, "combined_score": 0.0, "drum_rate_hz": np.nan}
    bi = int(np.argmax(props["prominences"]))
    peak_lag  = float(lags_in[peaks[bi]])
    peak_prom = float(props["prominences"][bi])

    h_lags, h_proms = [], []
    for k in (2, 3, 4):
        tgt = k * peak_lag
        if tgt > ac_t.max(): break
        win = 0.15 * tgt
        hm = (ac_t > tgt - win) & (ac_t < tgt + win)
        if hm.sum() < 3: continue
        hp, hpp = find_peaks(ac[hm], prominence=0.02)
        if len(hp) == 0: continue
        bii = int(np.argmax(hpp["prominences"]))
        h_lags.append(float(ac_t[hm][hp][bii]))
        h_proms.append(float(hpp["prominences"][bii]))

    return {
        "autocorr": ac, "autocorr_t": ac_t,
        "drum_prom":    peak_prom,
        "drum_lag":     peak_lag,
        "drum_rate_hz": 1.0 / peak_lag if peak_lag > 0 else np.nan,
        "harmonic_lags":  h_lags,
        "harmonic_proms": h_proms,
        "n_harmonics":    len(h_lags),
        "combined_score": peak_prom + 0.5 * sum(h_proms),
    }

def detect_continuation_calls(S_abs, freqs, sr,
                              band=HIGH_BAND_HZ,
                              hop=HOP_SPEC,
                              prom=HIGH_EVENT_PROM,
                              min_sep_s=HIGH_EVENT_MIN_S):
    in_band = (freqs >= band[0]) & (freqs <= band[1])
    env = S_abs[in_band].sum(axis=0)
    env_norm = env / (env.max() + 1e-10)
    min_dist = max(1, int(round(min_sep_s * sr / hop)))
    peaks, _ = find_peaks(env_norm, prominence=prom, distance=min_dist)
    env_t = np.arange(len(env)) * (hop / sr)
    return env_norm, env_t, peaks

# Sample bouts.
pool = bouts_meta[bouts_meta["bout_size"] >= MIN_BOUT_SIZE]
n_take = min(N_TO_SHOW, len(pool))
sample = pool.sample(n_take, random_state=0)
n_rows = (n_take + N_COLS - 1) // N_COLS

fig = plt.figure(figsize=(5.5 * N_COLS, 4.6 * n_rows))
outer_gs = fig.add_gridspec(n_rows, N_COLS, hspace=0.55, wspace=0.18)

results = {}
for i in range(n_take):
    r_block = i // N_COLS
    c       = i % N_COLS
    inner_gs = outer_gs[r_block, c].subgridspec(
        3, 1, height_ratios=[1.0, 1.5, 1.0], hspace=0.05,
    )
    ax_ac = fig.add_subplot(inner_gs[0])
    ax_hi = fig.add_subplot(inner_gs[1])
    ax_lo = fig.add_subplot(inner_gs[2])

    bout_id  = sample.index[i]
    bout_row = sample.iloc[i]
    bc = calls[calls["bout_id"] == bout_id].sort_values("start_time_file_sec")
    if bc.empty or bc["file_num"].nunique() != 1 or bc["channel"].nunique() != 1:
        for ax in (ax_ac, ax_hi, ax_lo):
            ax.set_axis_off()
        continue

    last_call = bc.iloc[-1]
    try:
        y, sr = load_call_slice(
            BASE_PROCESSED_AUDIO,
            last_call["date_folder"], last_call["exp"],
            last_call["channel"],     last_call["file_num"],
            last_call["stop_time_file_sec"],
            last_call["stop_time_file_sec"] + POST_WIN_SEC,
            pad_sec=0.0,
        )
    except Exception:
        for ax in (ax_ac, ax_hi, ax_lo):
            ax.set_axis_off()
        continue
    if len(y) < NFFT_SPEC:
        for ax in (ax_ac, ax_hi, ax_lo):
            ax.set_axis_off()
        continue
    actual_dur = len(y) / sr

    # --- Detectors ---
    r = detect_drum_tail_v2(y, sr)

    S = np.abs(librosa.stft(y.astype(np.float32),
                             n_fft=NFFT_SPEC, hop_length=HOP_SPEC, window="hann"))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=NFFT_SPEC)
    hi_env, hi_env_t, hi_peaks_idx = detect_continuation_calls(S, freqs, sr)
    n_calls = len(hi_peaks_idx)

    drum_score = r["combined_score"] if r is not None else 0.0
    drum_rate  = r["drum_rate_hz"]   if r is not None else np.nan

    # Count low-band drum impulses (peaks in the 0-3 kHz envelope) for the ratio check.
    drum_in_band = (freqs >= LOW_BAND_HZ[0]) & (freqs <= LOW_BAND_HZ[1])
    lo_env = S[drum_in_band].sum(axis=0)
    lo_env_norm = lo_env / (lo_env.max() + 1e-10)
    lo_min_dist = max(1, int(round(0.040 * sr / HOP_SPEC)))   # >= 40 ms apart
    lo_peaks_idx, _ = find_peaks(lo_env_norm, prominence=HIGH_EVENT_PROM, distance=lo_min_dist)
    n_drum_events = len(lo_peaks_idx)

    class_label, class_color = classify(drum_score, n_calls, n_drum_events)
    ratio_print = (n_calls / max(n_drum_events, 1)) if n_drum_events else 0.0
    results[i + 1] = {
        "drum_prom":      r["drum_prom"]      if r else 0,
        "n_harmonics":    r["n_harmonics"]    if r else 0,
        "combined_score": drum_score,
        "drum_rate_hz":   drum_rate,
        "n_calls":        n_calls,
        "n_drums":        n_drum_events,
        "ratio_c_d":      ratio_print,
        "class":          class_label,
    }

    # --- Autocorrelation panel ---
    if r is None:
        ax_ac.set_axis_off()
    else:
        ax_ac.plot(r["autocorr_t"], r["autocorr"], "-", color="navy", linewidth=1.0)
        ax_ac.axvspan(DRUM_LAG_RANGE_S[0], DRUM_LAG_RANGE_S[1],
                      color="green", alpha=0.10)
        if not np.isnan(r["drum_lag"]):
            ax_ac.axvline(r["drum_lag"], color="red", linewidth=1.2, alpha=0.85)
            for h_lag in r["harmonic_lags"]:
                ax_ac.axvline(h_lag, color="orange", linewidth=1.0,
                              alpha=0.75, linestyle="--")
        ax_ac.axhline(0, color="gray", linewidth=0.4)
        ax_ac.set_xlim(0, 2.0)
        ax_ac.set_ylim(-0.4, 1.05)
        ax_ac.set_xticklabels([])
        if c == 0:
            ax_ac.set_ylabel("AC", fontsize=8)
        rt_s = f"{drum_rate:.1f}Hz" if drum_rate and not np.isnan(drum_rate) else "—"
        ax_ac.set_title(
            f"#{i+1}  drum={drum_score:.2f}  calls={n_calls}/drums={n_drum_events} "
            f"(ratio={ratio_print:.2f})  [{class_label}]\n  rate={rt_s}",
            fontsize=9, loc="left", color=class_color, fontweight="bold",
            pad=2,
        )

    # --- Spectrograms ---
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    extent = [0, actual_dur, freqs[0] / 1000, freqs[-1] / 1000]

    ax_hi.imshow(S_db, aspect="auto", origin="lower", cmap="magma",
                 vmin=-55, vmax=-5, extent=extent)
    ax_hi.set_ylim(HIGH_BAND_HZ[0] / 1000, HIGH_BAND_HZ[1] / 1000)
    ax_hi.set_xticks([])
    # Overlay high-band envelope as a curve at the bottom of the alarm panel.
    ax_hi_twin = ax_hi.twinx()
    ax_hi_twin.plot(hi_env_t, hi_env, "-", color="cyan", linewidth=0.9, alpha=0.7)
    ax_hi_twin.set_ylim(0, 4)   # leave headroom
    ax_hi_twin.set_yticks([])
    # Mark detected high-band events with cyan ticks at top.
    for pi in hi_peaks_idx:
        ax_hi.axvline(hi_env_t[pi], color="cyan", linewidth=0.6, alpha=0.6)
    if c == 0:
        ax_hi.set_ylabel("kHz (alarm)", fontsize=8)

    ax_lo.imshow(S_db, aspect="auto", origin="lower", cmap="magma",
                 vmin=-55, vmax=-5, extent=extent)
    ax_lo.set_ylim(0, 3)
    if c == 0:
        ax_lo.set_ylabel("kHz (drum)", fontsize=8)
    if r_block == n_rows - 1:
        ax_lo.set_xlabel("time after last call (s)", fontsize=8)

    # Memory cleanup.
    del y, S, S_db, hi_env, hi_env_t
    if r is not None:
        r.pop("autocorr", None); r.pop("autocorr_t", None)
    gc.collect()

fig.suptitle(
    "Post-bout: drum detector + continuation-call detector  ·  "
    "title colour: green=drum_only, orange=drum+calls, red=calls_only, gray=quiet",
    y=1.005, fontsize=12,
)
fig.tight_layout()
save_fig(fig, "alarm_drum_continuation_combined")
plt.show()

post_bout_candidates = sample.reset_index().assign(
    candidate_num=range(1, len(sample) + 1)
)
summary = (
    pd.DataFrame([{"candidate_num": k, **v} for k, v in results.items()])
    .sort_values("combined_score", ascending=False)
    .reset_index(drop=True)
)
print("\nResults (sorted by drum combined_score):")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nClass counts:")
print(summary["class"].value_counts().to_string())


## Call duration trajectory within bouts

How does call duration change as a bout progresses? **Spaghetti plot**: one thin gray line per bout shows its own (paired) trajectory; the red line + envelope shows the population median ± IQR at each position.

Two caveats:
- **Selection bias at later positions.** Position `p` is computed only over bouts with at least `p` calls — so the right side of each panel reflects increasingly *long* (and therefore probably more intense) bouts. The legend reports the `n` of bouts contributing at the first and last visible position so the bias is visible.
- **Spaghetti is subsampled** (random `N_SPAGHETTI_MAX`) for visual clarity; the red median/IQR uses **all** bouts in the date.

In [ ]:
# Within-bout call duration trajectory, per date.
#
# Fan chart per date: median line + IQR (25-75%) band + 5-95% band. Each band
# is computed at every position across the bouts that reach that position
# (paired - each bout contributes its own trajectory to the stats).

MIN_SIZE = 5     # only "real" multi-call bouts (matches in_bout threshold)
MAX_POS  = 15    # cap the x-axis; most bouts don't reach this far

fig, axes = plt.subplots(
    len(DATES_TO_PLOT), 1,
    figsize=(11, 2.6 * len(DATES_TO_PLOT)),
    sharex=True, sharey=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = calls[
        (calls["date_folder"] == date)
        & (calls["bout_size"] >= MIN_SIZE)
        & (calls["bout_position"] <= MAX_POS)
    ]
    # Wide-format duration matrix: rows = bouts, cols = position, values = duration_sec.
    mat = sub.pivot(index="bout_id", columns="bout_position", values="duration_sec")
    if mat.empty:
        ax.set_axis_off()
        continue

    # Percentile bands and median - computed across ALL bouts at each position.
    p05    = mat.quantile(0.05, axis=0)
    p25    = mat.quantile(0.25, axis=0)
    median = mat.median(axis=0)
    p75    = mat.quantile(0.75, axis=0)
    p95    = mat.quantile(0.95, axis=0)
    n_per_pos = mat.notna().sum(axis=0)

    # Outer band: 5-95 percentile (light).
    ax.fill_between(median.index, p05, p95, color="red", alpha=0.10, label="5-95%")
    # Inner band: IQR (medium).
    ax.fill_between(median.index, p25, p75, color="red", alpha=0.25, label="IQR")
    # Median line (bold).
    ax.plot(median.index, median.values, "-o", color="darkred",
            linewidth=2, markersize=5, label="median")

    n_first = int(n_per_pos.iloc[0])
    n_last  = int(n_per_pos.iloc[-1])
    ax.set_title(
        f"{date}  ({mat.shape[0]:,} bouts >= {MIN_SIZE} calls;  "
        f"n at pos 1 = {n_first:,};  n at pos {int(median.index[-1])} = {n_last:,})",
        loc="left",
    )
    ax.set_ylabel("Call duration (s)")
    ax.legend(loc="upper right", fontsize=8)

# Tight linear y-axis - captures medians and most of the IQR/5-95 bands without
# wasting space on the extreme outlier tail.
axes[0].set_ylim(0, 0.25)
axes[-1].set_xticks(range(1, MAX_POS + 1))
axes[-1].set_xlabel("Position in bout")

fig.suptitle("Within-bout call duration trajectory per date")
fig.tight_layout()
save_fig(fig, "alarm_duration_trajectory_per_date")
plt.show()

## Acoustic feature trajectories within bouts

To capture the multi-feature signal from the spectrogram example (duration shrinks + upper harmonic disappears + call simplifies), we run `compute_features` on every call in a subsample of in_bout bouts. Then plot fan charts of each feature across `bout_position`, just like the duration trajectory above.

**Subsampling:** `compute_features` takes ~30–50 ms per call, so running on all 23 k alarm calls would be slow. We sample `N_BOUTS_PER_DATE` bouts per date (defaulting to 100), then keep all calls inside those bouts → roughly 2–3 k calls total → ~2 min runtime.

**Cluster only** — needs raw WAVs.

In [ ]:
# Extract vocalpy features for every call in a subsample of in_bout bouts.
# Subsample because compute_features takes ~30-50 ms/call - extracting all
# 23 k alarm calls would take ~20 min. 100 bouts/date * ~7 calls/bout ~ 2800
# calls total, ~2 min runtime.
#
# PAD_SEC: load each call with a small symmetric buffer of audio on each side.
# Mitigates the segmenter occasionally cutting a call's tail (and possibly
# leading edge). Features (FM, pitch, PSD) are then computed over the full
# call. `duration_s` is overwritten with the segmenter's ORIGINAL call
# duration so the buffer doesn't inflate it.

N_BOUTS_PER_DATE = 100
SEED             = 0
PAD_SEC          = 0.01    # 10 ms each side

if HOST != "Linux":
    raise RuntimeError("Need raw WAVs - run on the cluster.")

rng = np.random.default_rng(SEED)
sampled_bouts_parts = []
for date in DATES_TO_PLOT:
    pool = bouts_meta[
        (bouts_meta["date_folder"] == date)
        & (bouts_meta["bout_kind"] == "in_bout")
    ]
    if pool.empty:
        continue
    n = min(N_BOUTS_PER_DATE, len(pool))
    sample = pool.sample(n, random_state=int(rng.integers(0, 1_000_000)))
    sampled_bouts_parts.append(sample)

sampled_bouts = pd.concat(sampled_bouts_parts)
sampled_calls = calls[calls["bout_id"].isin(sampled_bouts.index)].copy()
print(f"Sampled {len(sampled_bouts):,} bouts across {len(DATES_TO_PLOT)} dates "
      f"-> {len(sampled_calls):,} calls to extract features for.")
print(f"Using pad_sec = {PAD_SEC*1000:.0f} ms on each side of each call.")

records = []
for i, row in enumerate(sampled_calls.itertuples(index=False)):
    y, sr = load_call_slice(
        BASE_PROCESSED_AUDIO,
        row.date_folder, row.exp, row.channel, row.file_num,
        row.start_time_file_sec, row.stop_time_file_sec,
        pad_sec=PAD_SEC,
    )
    feats = compute_features(y, sr)
    # Override: duration_s should reflect the SEGMENTER's call duration,
    # not the padded audio. Other features (FM, pitch, PSD) keep their
    # padded-audio values - that's the whole point of padding.
    feats["duration_s"] = row.stop_time_file_sec - row.start_time_file_sec
    records.append({
        "bout_id":       row.bout_id,
        "bout_position": row.bout_position,
        "bout_size":     row.bout_size,
        "date_folder":   row.date_folder,
        **feats,
    })
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(sampled_calls):,}")

features_df = pd.DataFrame(records)
print(f"\nfeatures_df: {features_df.shape[0]:,} rows x {features_df.shape[1]} cols")
print(f"Per-date call counts:")
print(features_df.groupby("date_folder").size())
features_df.head()

In [ ]:
# Join event-scale info onto features_df so the same analyses can run at
# either scale. After this cell, features_df has both:
#   bout_id,  bout_position,  bout_size       (already from the extraction)
#   event_id, event_position, event_size      (added here)
#
# Caveat: features_df was built from bout-sampled calls (100 in_bout bouts per
# date), so the events represented here are a non-random subset. Good enough
# for a quick look; if results are interesting, re-extract by sampling events
# instead of bouts.

# Idempotent: drop any pre-existing event_* columns before merging.
features_df = features_df.drop(
    columns=[c for c in ("event_id", "event_position", "event_size")
             if c in features_df.columns]
)

events_info = (
    calls[["bout_id", "bout_position", "event_id", "event_position", "event_size"]]
)
features_df = features_df.merge(
    events_info, on=["bout_id", "bout_position"], how="left",
)

print(f"features_df now has {features_df.shape[1]} cols (added event_id/position/size).")
print(f"All rows have event info: {features_df['event_position'].notna().all()}")
print(f"\nUnique events in features_df, per date:")
print(features_df.groupby("date_folder")["event_id"].nunique())

### Verify the FM-rising effect: example bouts

The feature-trajectory plot above showed `fm_median` rising with `bout_position` across all 4 dates. To check this isn't a measurement artifact: pick one random multi-call bout per date (10–15 calls), show every call's spectrogram, and the FM trajectory below.

If the trend is real, you should be able to *see* later calls having more in-call frequency modulation (curvier energy lines on the spectrogram) than earlier ones.

In [ ]:
# One example bout per date - random pick from bouts of size 10-15 in features_df.
# For each: a strip of mini-spectrograms (one per call) plus an FM trajectory below.
#
# Audio is loaded with PAD_SEC of buffer on each side, so we can spot cases
# where the segmenter cut a call's tail. Two thin white vertical lines on
# each spectrogram show the segmenter's [start, stop] - energy outside those
# lines is "missing call" (or noise) that the feature extraction would have
# otherwise omitted.

import librosa

if HOST != "Linux":
    raise RuntimeError("Need raw WAVs - run on the cluster.")

EXAMPLE_SEED   = 0
SIZE_RANGE     = (10, 15)
SPEC_FREQ_LIM  = 60       # kHz
PAD_SEC        = 0.01     # match the feature-extraction buffer

rng_ex = np.random.default_rng(EXAMPLE_SEED)

example_bout_ids = []
for date in DATES_TO_PLOT:
    pool = features_df[
        (features_df["date_folder"] == date)
        & (features_df["bout_size"] >= SIZE_RANGE[0])
        & (features_df["bout_size"] <= SIZE_RANGE[1])
    ]["bout_id"].unique()
    if len(pool) == 0:
        print(f"  {date}: no bouts in size range {SIZE_RANGE}")
        continue
    example_bout_ids.append(int(rng_ex.choice(pool)))

for bout_id in example_bout_ids:
    bout_calls = (
        calls[calls["bout_id"] == bout_id]
        .sort_values("bout_position")
        .reset_index(drop=True)
    )
    n_calls = len(bout_calls)
    first   = bout_calls.iloc[0]
    date    = first["date_folder"]
    exp     = int(first["exp"])
    file_num     = int(first["file_num"])
    start_sec    = float(first["start_time_file_sec"])
    location     = first["assigned_location"]

    fig = plt.figure(figsize=(1.4 * n_calls + 1, 4.5))
    gs  = fig.add_gridspec(2, n_calls, height_ratios=[3, 1], hspace=0.6)
    spec_axes = [fig.add_subplot(gs[0, i]) for i in range(n_calls)]
    fm_ax     = fig.add_subplot(gs[1, :])

    for i, row in enumerate(bout_calls.itertuples(index=False)):
        # Load with buffer; we'll mark the segmenter's original boundaries.
        y, sr = load_call_slice(
            BASE_PROCESSED_AUDIO,
            row.date_folder, row.exp, row.channel, row.file_num,
            row.start_time_file_sec, row.stop_time_file_sec,
            pad_sec=PAD_SEC,
        )
        stft = librosa.stft(
            y, n_fft=SAT_PARAMS["n_fft"],
            hop_length=SAT_PARAMS["hop_length"], window="hann",
        )
        spec_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

        # The padded audio's time axis: 0 is the start of the padded read,
        # which (when not clipped by the file edge) is start_sec - PAD_SEC.
        # Compute the actual left-pad in ms (might be < PAD_SEC near file start).
        n_samples = len(y)
        total_ms  = n_samples / sr * 1000
        # Segmenter's call boundaries within the padded clip:
        call_dur_ms  = (row.stop_time_file_sec - row.start_time_file_sec) * 1000
        left_pad_ms  = (total_ms - call_dur_ms) * (PAD_SEC / (2 * PAD_SEC))  # equal split unless clipped
        # In practice the equal-split assumption is good enough away from file edges;
        # just use PAD_SEC*1000 directly:
        left_pad_ms  = PAD_SEC * 1000
        right_pad_ms = total_ms - call_dur_ms - left_pad_ms
        call_onset_ms  = left_pad_ms
        call_offset_ms = left_pad_ms + call_dur_ms

        ax = spec_axes[i]
        ax.imshow(
            spec_db, origin="lower", aspect="auto",
            extent=[0, total_ms, 0, sr / 2 / 1000],
            cmap="magma", vmin=-80, vmax=0,
        )
        ax.set_ylim(0, SPEC_FREQ_LIM)
        # Two white vertical lines = segmenter's [start, stop].
        # Energy outside these lines is in the buffer.
        ax.axvline(call_onset_ms,  color="white", lw=0.8, alpha=0.7)
        ax.axvline(call_offset_ms, color="white", lw=0.8, alpha=0.7)

        if i == 0:
            ax.set_title(f"{i + 1}", fontsize=9)
        else:
            ici_ms = row.ici_s * 1000
            ax.set_title(f"{i + 1}\nΔ {ici_ms:.0f} ms", fontsize=9)
        ax.set_xlabel(f"{row.start_time_file_sec:.2f}s", fontsize=8)
        ax.set_xticks([])
        if i > 0:
            ax.set_yticks([])
    spec_axes[0].set_ylabel("kHz")

    fm_row = (
        features_df[features_df["bout_id"] == bout_id]
        .sort_values("bout_position")[["bout_position", "fm_median"]]
    )
    fm_ax.bar(
        fm_row["bout_position"].values, fm_row["fm_median"].values,
        color="darkred", edgecolor="white", linewidth=0.5,
    )
    fm_ax.set_xticks(fm_row["bout_position"].values)
    fm_ax.set_xlim(0.5, n_calls + 0.5)
    fm_ax.set_ylim(0, max(1.0, fm_row["fm_median"].max() * 1.15))
    fm_ax.set_ylabel("fm_median (rad)")
    fm_ax.set_xlabel("position in bout")

    fig.suptitle(
        f"{date}  ·  bout {bout_id}  ·  {n_calls} calls   |   "
        f"exp {exp}  ·  file {file_num:03d}  ·  start {start_sec:.2f} s  ·  {location}    "
        f"(white lines = segmenter onset/offset; pad = {int(PAD_SEC*1000)} ms each side)",
        y=1.02, fontsize=11,
    )
    fig.tight_layout()
    plt.show()

## Duration vs pitch curvature, colored by position

Two of the position-sensitive features plotted against each other. If the within-bout trajectory is "flat-and-long → curvy-and-short", we should see calls form a diagonal cloud:

- **Top-right** (long duration, less negative / near-zero curvature) = position 1 calls (flat).
- **Bottom-left** (short duration, strongly negative curvature) = late-position calls (curvy ∩-shape).

The black line connects per-position medians so the average trajectory is visible across the cloud.

In [ ]:
# Scatter: duration_s (x) vs pitch_curvature_hz_per_call2 (y), colored by position.
# Three layers (back-to-front):
#   1. Thin gray lines connecting calls within the same bout/event (subsampled).
#   2. Cloud of all calls, colored by position-in-scale.
#   3. Per-position medians + connecting line.
#
# Set SCALE = "bout" or "event" to switch which grouping the analysis uses.

SCALE = "event"     # <-- toggle between "bout" and "event"

id_col, pos_col, size_col = f"{SCALE}_id", f"{SCALE}_position", f"{SCALE}_size"

MAX_POS_SCATTER = 15
CAP_DUR_MS      = 250
N_GROUP_LINES   = 40        # subsampled per-group trajectories per panel
LINE_SEED       = 1

cmap = plt.get_cmap("viridis")
rng_line = np.random.default_rng(LINE_SEED)

fig, axes = plt.subplots(
    1, len(DATES_TO_PLOT),
    figsize=(3.6 * len(DATES_TO_PLOT), 4.2),
    sharex=True, sharey=True,
)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = features_df[
        (features_df["date_folder"] == date)
        & (features_df[pos_col] <= MAX_POS_SCATTER)
        & features_df["pitch_curvature_hz_per_call2"].notna()
    ].copy()
    sub["duration_ms"] = sub["duration_s"] * 1000
    sub = sub[sub["duration_ms"] <= CAP_DUR_MS]
    if sub.empty:
        ax.set_axis_off()
        continue

    # --- Layer 1: thin per-group lines (subsampled). ---
    group_ids = sub[id_col].unique()
    n_pick = min(N_GROUP_LINES, len(group_ids))
    picked = rng_line.choice(group_ids, size=n_pick, replace=False)
    for gid in picked:
        gdata = sub[sub[id_col] == gid].sort_values(pos_col)
        if len(gdata) < 2:
            continue
        ax.plot(
            gdata["duration_ms"], gdata["pitch_curvature_hz_per_call2"],
            "-", color="gray", linewidth=0.4, alpha=0.25, zorder=2,
        )

    # --- Layer 2: cloud of every call (colored by position). ---
    sc = ax.scatter(
        sub["duration_ms"], sub["pitch_curvature_hz_per_call2"],
        c=sub[pos_col],
        cmap=cmap, vmin=1, vmax=MAX_POS_SCATTER,
        s=8, alpha=0.45, edgecolor="none", zorder=4,
    )

    # --- Layer 3: per-position medians + connecting line. ---
    medians = []
    for pos in range(1, MAX_POS_SCATTER + 1):
        ps = sub[sub[pos_col] == pos]
        if len(ps) < 5:
            continue
        medians.append({
            "position":    pos,
            "duration_ms": ps["duration_ms"].median(),
            "curvature":   ps["pitch_curvature_hz_per_call2"].median(),
        })
    med_df = pd.DataFrame(medians)
    if not med_df.empty:
        ax.plot(med_df["duration_ms"], med_df["curvature"],
                "-", color="black", linewidth=1.5, alpha=0.85, zorder=9)
        ax.scatter(
            med_df["duration_ms"], med_df["curvature"],
            c=med_df["position"], cmap=cmap, vmin=1, vmax=MAX_POS_SCATTER,
            s=90, edgecolor="black", linewidth=0.8, zorder=10,
        )

    ax.set_title(f"{date}  (n={len(sub):,} calls, {n_pick} {SCALE} lines)",
                 fontsize=10, loc="left")
    ax.set_xlabel("duration (ms)")
    ax.axhline(0, color="gray", linestyle=":", linewidth=0.6, alpha=0.5)

axes[0].set_ylabel("pitch_curvature_hz_per_call2")

cbar = fig.colorbar(sc, ax=axes, label=f"position in {SCALE}", pad=0.02, shrink=0.85)
cbar.set_ticks(range(1, MAX_POS_SCATTER + 1, 2))

fig.suptitle(
    f"Duration vs pitch curvature  ·  per-{SCALE} lines + per-position medians",
    y=1.02, fontsize=11,
)
save_fig(fig, f"alarm_duration_vs_curvature_per_{SCALE}")
plt.show()

## Binary decoder: is this call position 1?

Cleanest framing of "first-call-is-special": a binary classifier. For each call, can we predict — from acoustic features alone — whether it's the first call of its event (or bout)?

Why this should outperform the regression:
- The visual pattern in our scatter plots was a step, not a gradient: position 1 vs everything else. A binary target matches that.
- Position 1 has ~10% prevalence (one call per multi-call group), so we use `class_weight="balanced"` so the loss isn't dominated by the majority class.
- AUC is the right metric — threshold-free, robust to class imbalance.

Expected coefficient signs (inverted from the position-regression):
- `duration_s` **positive** (longer → position 1)
- `pitch_curvature_hz_per_call2` **positive** (less negative = flatter → position 1)
- `fm_median` **negative** (lower FM → position 1)

In [ ]:
# Binary classifier: is this call position 1 of its bout/event?

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve

SCALE   = "event"     # <-- toggle between "bout" and "event"
MAX_POS = 15

id_col, pos_col = f"{SCALE}_id", f"{SCALE}_position"

FEATURE_COLS = [
    "duration_s",
    "peak_freq_hz",
    "pitch_curvature_hz_per_call2",
    "entropy_s",
    "fm_median",
]

keep = features_df[
    features_df[pos_col] <= MAX_POS
][FEATURE_COLS + [pos_col, id_col, "date_folder"]].dropna()

X      = keep[FEATURE_COLS].values
y      = (keep[pos_col] == 1).astype(int).values    # binary target
groups = keep[id_col].values

print(f"SCALE = {SCALE!r}.  {len(keep):,} calls, {y.sum():,} positives "
      f"({y.mean()*100:.1f}%), {len(set(groups)):,} {SCALE}s.")

# Cross-validated logistic regression, grouped by id_col.
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
cv = GroupKFold(n_splits=5)
proba = cross_val_predict(
    pipe, X, y, cv=cv, groups=groups, method="predict_proba",
)[:, 1]

# Overall AUC + per-date AUCs.
auc_overall = roc_auc_score(y, proba)
print(f"\nCross-validated AUC overall = {auc_overall:.3f}")

print("\nPer-date AUC:")
per_date_aucs = []
for date in DATES_TO_PLOT:
    mask = (keep["date_folder"] == date).values
    if mask.sum() < 10 or y[mask].sum() == 0:
        continue
    auc_d = roc_auc_score(y[mask], proba[mask])
    n     = mask.sum()
    n_pos = y[mask].sum()
    per_date_aucs.append((date, auc_d, n, n_pos))
    print(f"  {date}: AUC = {auc_d:.3f}  (n = {n:,}, positives = {n_pos:,})")

# Fit on all data for interpretable coefficients.
pipe.fit(X, y)
coefs = pipe.named_steps["logreg"].coef_.ravel()
print(f"\nStandardized logistic coefficients (positive = predicts position 1):")
for col, c in sorted(zip(FEATURE_COLS, coefs), key=lambda p: -abs(p[1])):
    print(f"  {col:35s} {c:+.3f}")

# Plots.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5),
                         gridspec_kw={"width_ratios": [1.0, 1.0]})

# Left: ROC curve, with per-date curves overlaid faintly.
ax = axes[0]
for date, _, _, _ in per_date_aucs:
    mask = (keep["date_folder"] == date).values
    fpr_d, tpr_d, _ = roc_curve(y[mask], proba[mask])
    ax.plot(fpr_d, tpr_d, linewidth=1.0, alpha=0.5, label=f"{date} (AUC={per_date_aucs[[d[0] for d in per_date_aucs].index(date)][1]:.2f})")
fpr, tpr, _ = roc_curve(y, proba)
ax.plot(fpr, tpr, color="black", linewidth=2.5, label=f"pooled (AUC={auc_overall:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.4, label="chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC: position-1 vs rest  ({SCALE} scale)", loc="left", fontsize=10)
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.grid(alpha=0.2)

# Right: coefficient bar chart.
ax = axes[1]
order = np.argsort(-np.abs(coefs))
sorted_cols  = [FEATURE_COLS[i] for i in order]
sorted_coefs = coefs[order]
bar_colors = ["#1f77b4" if c < 0 else "#d62728" for c in sorted_coefs]
ax.barh(range(len(sorted_coefs)), sorted_coefs, color=bar_colors)
ax.set_yticks(range(len(sorted_coefs)))
ax.set_yticklabels(sorted_cols, fontsize=9)
ax.axvline(0, color="black", linestyle="--", linewidth=0.6)
ax.set_xlabel("standardized logistic coefficient")
ax.set_title("Feature importance\n(red = predicts position 1)", loc="left", fontsize=10)
ax.invert_yaxis()

fig.tight_layout()
save_fig(fig, f"alarm_decoder_binary_position1_{SCALE}")
plt.show()

## First call vs N-th call: violin slopegraph

Paired comparison: every group (event or bout, depending on `SCALE`) that has both a position-1 call and a position-`COMPARE_POS` call contributes one paired observation. Violins show distributions; thin gray lines connect the two halves of each pair; per-panel Wilcoxon p is signed-rank on the paired differences.

Two knobs:
- `SCALE` — `"event"` (coarser, fewer groups but each is longer) vs `"bout"` (finer, more groups, each shorter).
- `COMPARE_POS` — which position to compare against.


In [ ]:
# Violin slopegraph: first call vs Nth call, paired within group.
# Self-contained (does not depend on the now-deleted Version-A cell).

from scipy.stats import wilcoxon

# ---- config ----
SCALE       = "event"   # "event" or "bout"
COMPARE_POS = 10        # change to 5 / 15 / etc.
ID_COL  = f"{SCALE}_id"
POS_COL = f"{SCALE}_position"

FEATURES_VIOLIN = [
    ("Duration (ms)",              "duration_s",                   lambda v: v * 1000),
    ("Pitch curvature (Hz/call²)", "pitch_curvature_hz_per_call2", lambda v: v),
    ("FM median (rad)",            "fm_median",                    lambda v: v),
]

def stars(p):
    if np.isnan(p): return ""
    if p < 0.001:   return "***"
    if p < 0.01:    return "**"
    if p < 0.05:    return "*"
    return "n.s."

# ---- Build paired table: one row per group that has BOTH pos-1 and pos-N ----
records = []
for group_id, gdata in features_df.groupby(ID_COL):
    g_indexed = gdata.set_index(POS_COL)
    if 1 not in g_indexed.index or COMPARE_POS not in g_indexed.index:
        continue
    first_row = g_indexed.loc[1]
    nth_row   = g_indexed.loc[COMPARE_POS]
    if isinstance(first_row, pd.DataFrame): first_row = first_row.iloc[0]
    if isinstance(nth_row,   pd.DataFrame): nth_row   = nth_row.iloc[0]
    rec = {ID_COL: group_id, "date_folder": gdata["date_folder"].iloc[0]}
    for _, col, scale in FEATURES_VIOLIN:
        rec[f"{col}_first"] = scale(first_row[col])
        rec[f"{col}_nth"]   = scale(nth_row[col])
    records.append(rec)
paired_B = pd.DataFrame(records).dropna()
print(f"{SCALE}-scale, first vs pos {COMPARE_POS}: {len(paired_B)} {SCALE}s with both positions.")

# ---- Plot ----
fig, axes = plt.subplots(
    len(FEATURES_VIOLIN), len(DATES_TO_PLOT),
    figsize=(2.6 * len(DATES_TO_PLOT), 2.8 * len(FEATURES_VIOLIN)),
    sharey="row",
)

for r, (ylabel, col, _) in enumerate(FEATURES_VIOLIN):
    for c, date in enumerate(DATES_TO_PLOT):
        ax = axes[r, c]
        sub = paired_B[paired_B["date_folder"] == date]
        if sub.empty:
            ax.set_axis_off()
            continue
        firsts = sub[f"{col}_first"].values
        nths   = sub[f"{col}_nth"].values

        parts = ax.violinplot(
            [firsts, nths], positions=[0, 1],
            widths=0.75, showmeans=False, showmedians=True,
        )
        for pc, color in zip(parts["bodies"], ["#d62728", "#1f77b4"]):
            pc.set_facecolor(color)
            pc.set_alpha(0.45)

        # Paired connectors + endpoint dots.
        for f_val, n_val in zip(firsts, nths):
            ax.plot([0, 1], [f_val, n_val], "-",
                    color="gray", alpha=0.2, linewidth=0.4, zorder=2)
        ax.scatter([0]*len(firsts), firsts, s=8,
                   color="#d62728", alpha=0.5, zorder=3, edgecolor="none")
        ax.scatter([1]*len(nths), nths, s=8,
                   color="#1f77b4", alpha=0.5, zorder=3, edgecolor="none")

        diff = firsts - nths
        try:
            _, p = wilcoxon(diff)
        except ValueError:
            p = float("nan")
        med_diff = np.median(diff)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(["pos 1", f"pos {COMPARE_POS}"], fontsize=9)
        ax.set_xlim(-0.6, 1.6)
        if c == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        if r == 0:
            ax.set_title(f"{date}  (n={len(sub)})", fontsize=10)
        ax.text(0.5, 0.97,
                f"Δmed = {med_diff:+.2f}\np = {p:.1e} {stars(p)}",
                transform=ax.transAxes, ha="center", va="top", fontsize=8)

fig.suptitle(f"First call vs call at position {COMPARE_POS}  ({SCALE} scale)",
             y=1.02, fontsize=12)
fig.tight_layout()
save_fig(fig, f"alarm_violin_first_vs_pos{COMPARE_POS}_{SCALE}")
plt.show()


### Same, pooled across all dates

Same violin slopegraph (first call vs N-th call, paired within event), but pooling *all* dates into a single panel per feature. Cleaner for a one-glance "first call is different" slide; the per-date breakdown above is still the right figure if you want to show consistency across cohorts.

In [ ]:
# Violin slopegraph pooled across dates - one panel per feature.
# Standalone: rebuilds paired_B locally so it doesn't depend on the per-date cell.

from scipy.stats import wilcoxon

SCALE       = "event"
COMPARE_POS = 10
ID_COL  = f"{SCALE}_id"
POS_COL = f"{SCALE}_position"

FEATURES_VIOLIN = [
    ("Duration (ms)",              "duration_s",                   lambda v: v * 1000),
    ("Pitch curvature (Hz/call²)", "pitch_curvature_hz_per_call2", lambda v: v),
    ("FM median (rad)",            "fm_median",                    lambda v: v),
]

def stars(p):
    if np.isnan(p): return ""
    if p < 0.001:   return "***"
    if p < 0.01:    return "**"
    if p < 0.05:    return "*"
    return "n.s."

records = []
for group_id, gdata in features_df.groupby(ID_COL):
    g_indexed = gdata.set_index(POS_COL)
    if 1 not in g_indexed.index or COMPARE_POS not in g_indexed.index:
        continue
    first_row = g_indexed.loc[1]
    nth_row   = g_indexed.loc[COMPARE_POS]
    if isinstance(first_row, pd.DataFrame): first_row = first_row.iloc[0]
    if isinstance(nth_row,   pd.DataFrame): nth_row   = nth_row.iloc[0]
    rec = {ID_COL: group_id}
    for _, col, scale in FEATURES_VIOLIN:
        rec[f"{col}_first"] = scale(first_row[col])
        rec[f"{col}_nth"]   = scale(nth_row[col])
    records.append(rec)
paired_pooled = pd.DataFrame(records).dropna()
print(f"{SCALE}-scale, first vs pos {COMPARE_POS} pooled: "
      f"{len(paired_pooled)} {SCALE}s with both positions.")

# ---- Plot: 1 row x len(FEATURES) cols ----
fig, axes = plt.subplots(
    1, len(FEATURES_VIOLIN),
    figsize=(3.4 * len(FEATURES_VIOLIN), 4.0),
)
if len(FEATURES_VIOLIN) == 1:
    axes = [axes]

for ax, (ylabel, col, _) in zip(axes, FEATURES_VIOLIN):
    firsts = paired_pooled[f"{col}_first"].values
    nths   = paired_pooled[f"{col}_nth"].values
    if len(firsts) == 0:
        ax.set_axis_off()
        continue

    parts = ax.violinplot(
        [firsts, nths], positions=[0, 1],
        widths=0.75, showmeans=False, showmedians=True,
    )
    for pc, color in zip(parts["bodies"], ["#d62728", "#1f77b4"]):
        pc.set_facecolor(color); pc.set_alpha(0.45)

    # Paired connectors + endpoint dots.
    for f_val, n_val in zip(firsts, nths):
        ax.plot([0, 1], [f_val, n_val], "-",
                color="gray", alpha=0.15, linewidth=0.3, zorder=2)
    ax.scatter([0]*len(firsts), firsts, s=8,
               color="#d62728", alpha=0.4, zorder=3, edgecolor="none")
    ax.scatter([1]*len(nths), nths, s=8,
               color="#1f77b4", alpha=0.4, zorder=3, edgecolor="none")

    diff = firsts - nths
    try:
        _, p = wilcoxon(diff)
    except ValueError:
        p = float("nan")
    med_diff = np.median(diff)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["pos 1", f"pos {COMPARE_POS}"], fontsize=10)
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(
        f"Δmed = {med_diff:+.2f}\np = {p:.1e} {stars(p)}   (n={len(firsts)})",
        fontsize=10,
    )
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"First call vs call at position {COMPARE_POS} - pooled ({SCALE} scale)",
    y=1.02, fontsize=12,
)
fig.tight_layout()
save_fig(fig, f"alarm_violin_first_vs_pos{COMPARE_POS}_{SCALE}_pooled")
plt.show()


### Sweep N: which position-N gives the biggest contrast with position 1?

Instead of picking one N by eye, compute the **effect size (Cohen's d)** between
pos-1 calls and pos-N calls for every N in 2-15, separately for each feature.
Plot |Cohen's d| vs N; peak N tells you where the feature reaches its largest
opener-vs-mid-event separation.

In [ ]:
# Sweep N=2..15 for the paired comparison and pick the most informative
# position per feature. Uses *paired* within-group comparisons (same convention
# as the violin cell above).

from scipy.stats import wilcoxon

SCALE = "event"   # match the violin cell - "event" or "bout"
ID_COL  = f"{SCALE}_id"
POS_COL = f"{SCALE}_position"

SWEEP_FEATURES = [
    "duration_s",
    "pitch_curvature_hz_per_call2",
    "fm_median",
    "fm_early",
    "fm_late",
]
N_VALUES = list(range(2, 16))

records = []
for feat in SWEEP_FEATURES:
    for n in N_VALUES:
        diffs = []
        for group_id, gdata in features_df.groupby(ID_COL):
            g = gdata.set_index(POS_COL)
            if 1 not in g.index or n not in g.index:
                continue
            r1, rn = g.loc[1], g.loc[n]
            if isinstance(r1, pd.DataFrame): r1 = r1.iloc[0]
            if isinstance(rn, pd.DataFrame): rn = rn.iloc[0]
            v1, vn = r1[feat], rn[feat]
            if pd.isna(v1) or pd.isna(vn):
                continue
            diffs.append(vn - v1)
        if len(diffs) < 5:
            continue
        diffs = np.array(diffs)
        sd = diffs.std(ddof=1)
        d_paired = diffs.mean() / sd if sd > 0 else np.nan
        try:
            _, p = wilcoxon(diffs)
        except ValueError:
            p = np.nan
        records.append({
            "feature":   feat,
            "N":         n,
            "cohen_dz":  d_paired,
            "abs_dz":    abs(d_paired),
            "p":         p,
            "n_groups":  len(diffs),
        })

sweep_df = pd.DataFrame(records)

# ---- Plot ----
fig, (ax, ax_n) = plt.subplots(
    2, 1, figsize=(10, 5.4),
    gridspec_kw={"height_ratios": [3, 1]}, sharex=True,
)

feat_colors = dict(zip(SWEEP_FEATURES, plt.cm.tab10(range(len(SWEEP_FEATURES)))))
for feat in SWEEP_FEATURES:
    sub = sweep_df[sweep_df["feature"] == feat].sort_values("N")
    if sub.empty:
        continue
    ax.plot(sub["N"], sub["abs_dz"], "-o", color=feat_colors[feat],
            label=feat, linewidth=1.5, markersize=5)
    peak_row = sub.loc[sub["abs_dz"].idxmax()]
    ax.scatter(peak_row["N"], peak_row["abs_dz"],
               color=feat_colors[feat], s=140, edgecolor="black",
               linewidth=1.2, zorder=10)
    ax.annotate(f"N={int(peak_row['N'])}",
                (peak_row["N"], peak_row["abs_dz"]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=8, color=feat_colors[feat], va="center")

ax.set_ylabel("|Cohen's $d_z$|  (paired, pos-N vs pos-1)")
ax.set_title(f"Paired effect size of pos-1 vs pos-N across the {SCALE}")
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)

n_by_N = sweep_df.groupby("N")["n_groups"].max()
ax_n.bar(n_by_N.index, n_by_N.values, color="gray", alpha=0.7)
ax_n.set_xlabel(f"N (position compared against position 1)")
ax_n.set_ylabel(f"# paired {SCALE}s")
ax_n.set_xticks(N_VALUES)
ax_n.grid(axis="y", alpha=0.3)

fig.tight_layout()
save_fig(fig, f"alarm_pos1_vs_posN_effect_sweep_paired_{SCALE}")
plt.show()

print(f"\nBest N per feature ({SCALE} scale, largest paired |Cohen's d_z|):")
best = sweep_df.loc[sweep_df.groupby("feature")["abs_dz"].idxmax()]
print(best[["feature", "N", "cohen_dz", "p", "n_groups"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3g}"))


## Event-opener vs mid-event bout opener: state or motor?

The bout-scale effect being roughly half the event-scale effect suggests the
"first call is special" phenomenon isn't a generic motor reset that fires at
the start of every bout. To test directly:

For each event with **>= 2 bouts**, compare:
- **Red**: the event-opener (first call of the first bout, i.e. event-position 1).
- **Orange**: the first call of a later bout in the same event (paired within
  event by taking the median across all `bout_position=1` calls at
  `event_position > 1`).

Both are "first call of a bout" in motor terms. Only the red one is also the
"first call of the event." If they're indistinguishable -> motor reset. If red
is reliably more opener-like -> event-state gates the effect.

**Duration only here** — uses the full `calls` table (every call, no sampling)
so N is maximal. If duration shows the effect, we can re-extract features for
this event subset to test pitch curvature and FM too.


In [ ]:
# Event-opener vs mid-event bout-opener — duration-only version.
# Uses the full `calls` table (no feature-extraction sampling), so N is maximal.
# Pitch curvature and FM are skipped here; if duration shows a signal we can
# re-extract features for events with >=2 bouts to test those too.

from scipy.stats import wilcoxon

def stars(p):
    if np.isnan(p): return ""
    if p < 0.001:   return "***"
    if p < 0.01:    return "**"
    if p < 0.05:    return "*"
    return "n.s."

# Compute call duration in seconds if not already present.
if "duration_s" not in calls.columns:
    calls = calls.assign(
        duration_s=calls["stop_time_file_sec"] - calls["start_time_file_sec"]
    )

# Build per-event paired rows using duration only.
records = []
for ev_id, ev_data in calls.groupby("event_id"):
    opener_rows = ev_data[(ev_data["bout_position"] == 1) &
                          (ev_data["event_position"] == 1)]
    mid_bout_rows = ev_data[(ev_data["bout_position"] == 1) &
                            (ev_data["event_position"] > 1)]
    if opener_rows.empty or mid_bout_rows.empty:
        continue
    opener = opener_rows.iloc[0]
    records.append({
        "event_id":    ev_id,
        "date_folder": ev_data["date_folder"].iloc[0],
        "opener_ms":   opener["duration_s"] * 1000,
        "midbout_ms":  mid_bout_rows["duration_s"].median() * 1000,
    })
paired_state = pd.DataFrame(records).dropna()
print(f"Events with both event-opener and >= 1 mid-event bout: {len(paired_state)}")
print(f"Per date:\n{paired_state.groupby('date_folder').size().to_string()}")

# ---- Plot: 1 row (duration), 4 cols (dates) ----
fig, axes = plt.subplots(1, len(DATES_TO_PLOT),
                         figsize=(2.8 * len(DATES_TO_PLOT), 3.5),
                         sharey=True)

for ax, date in zip(axes, DATES_TO_PLOT):
    sub = paired_state[paired_state["date_folder"] == date]
    if sub.empty:
        ax.set_axis_off()
        continue
    openers  = sub["opener_ms"].values
    midbouts = sub["midbout_ms"].values

    parts = ax.violinplot(
        [openers, midbouts], positions=[0, 1],
        widths=0.75, showmeans=False, showmedians=True,
    )
    for pc, color in zip(parts["bodies"], ["#d62728", "#ff7f0e"]):
        pc.set_facecolor(color); pc.set_alpha(0.45)

    for o_val, m_val in zip(openers, midbouts):
        ax.plot([0, 1], [o_val, m_val], "-",
                color="gray", alpha=0.2, linewidth=0.4, zorder=2)
    ax.scatter([0]*len(openers), openers, s=8,
               color="#d62728", alpha=0.5, zorder=3, edgecolor="none")
    ax.scatter([1]*len(midbouts), midbouts, s=8,
               color="#ff7f0e", alpha=0.5, zorder=3, edgecolor="none")

    diff = openers - midbouts
    try:
        _, p = wilcoxon(diff)
    except ValueError:
        p = float("nan")
    med_diff = np.median(diff)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["event\nopener", "mid-evt\nbout-1"], fontsize=9)
    ax.set_xlim(-0.6, 1.6)
    ax.set_title(f"{date}  (n={len(sub)})", fontsize=10)
    ax.text(0.5, 0.97,
            f"Δmed = {med_diff:+.1f} ms\np = {p:.1e} {stars(p)}",
            transform=ax.transAxes, ha="center", va="top", fontsize=9)

axes[0].set_ylabel("Duration (ms)", fontsize=10)
fig.suptitle(
    "Event-opener vs mid-event bout-opener  ·  duration only  ·  paired within event",
    y=1.02, fontsize=12,
)
fig.tight_layout()
save_fig(fig, "alarm_violin_event_opener_vs_mid_event_bout_duration")
plt.show()


### Same test, all three features: re-extract for events with >=2 bouts

The duration-only result is solid. To check whether pitch curvature and FM
show the same state-gating, we need acoustic features for the event-opener
and mid-event bout-1 calls — which are mostly absent from `features_df` (random
bout sampling). This cell re-extracts features for exactly those calls
(targeted, much smaller than the original ~20-min full-corpus job), then
runs the same paired comparison on three features at once.

Expected runtime: ~2-3 minutes (extracts features for ~3-4 k calls).

In [ ]:
# Targeted feature extraction for the state-vs-motor test.
# Only the calls we need: event-opener (1 per event) + every mid-event bout-1
# call (1+ per event with >= 2 bouts).

from vocalization_analysis.acoustic_features import compute_features, load_call_slice

if HOST != "Linux":
    raise RuntimeError("Need raw WAVs - run on the cluster.")

# Identify the calls to extract.
target_mask = (
    (calls["bout_position"] == 1)
    & (
        (calls["event_position"] == 1)                                  # openers
        | (calls.groupby("event_id")["bout_position"].transform("count") > 1)  # placeholder
    )
)
# Cleaner: just take all calls with bout_position==1; we'll filter by event later.
target_calls = calls[calls["bout_position"] == 1].copy()

# Keep only events that have BOTH an event-opener and at least one mid-event bout-1.
have_opener = set(target_calls.loc[target_calls["event_position"] == 1, "event_id"])
have_midbout = set(target_calls.loc[target_calls["event_position"] > 1, "event_id"])
events_with_both = have_opener & have_midbout
target_calls = target_calls[target_calls["event_id"].isin(events_with_both)].copy()

print(f"Events with both event-opener and >=1 mid-event bout: {len(events_with_both)}")
print(f"Calls to extract features for: {len(target_calls):,}")
print(f"Estimated runtime at 40 ms/call: ~{len(target_calls) * 0.04 / 60:.1f} min")

PAD_SEC = 0.01
records = []
for i, row in enumerate(target_calls.itertuples(index=False)):
    try:
        y, sr = load_call_slice(
            BASE_PROCESSED_AUDIO,
            row.date_folder, row.exp, row.channel, row.file_num,
            row.start_time_file_sec, row.stop_time_file_sec,
            pad_sec=PAD_SEC,
        )
        feats = compute_features(y, sr)
        feats["duration_s"] = row.stop_time_file_sec - row.start_time_file_sec
        records.append({
            "event_id":       row.event_id,
            "event_position": row.event_position,
            "bout_id":        row.bout_id,
            "bout_position":  row.bout_position,
            "date_folder":    row.date_folder,
            **feats,
        })
    except Exception as e:
        if i < 5:
            print(f"  failed at i={i}: {e}")
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(target_calls):,}")

features_state_df = pd.DataFrame(records)
print(f"\nExtracted features: {len(features_state_df):,} rows x "
      f"{features_state_df.shape[1]} cols")


**Plot**: same recipe as the duration-only version, now with three features.

In [ ]:
# Paired state-vs-motor violin slopegraph on the targeted extraction.
# Uses features_state_df from the extraction cell above.

from scipy.stats import wilcoxon

FEATURES_STATE_VIOLIN = [
    ("Duration (ms)",              "duration_s",                   lambda v: v * 1000),
    ("Pitch curvature (Hz/call²)", "pitch_curvature_hz_per_call2", lambda v: v),
    ("FM median (rad)",            "fm_median",                    lambda v: v),
]

def stars(p):
    if np.isnan(p): return ""
    if p < 0.001:   return "***"
    if p < 0.01:    return "**"
    if p < 0.05:    return "*"
    return "n.s."

records_p = []
for ev_id, ev_data in features_state_df.groupby("event_id"):
    opener_rows = ev_data[ev_data["event_position"] == 1]
    mid_bout_rows = ev_data[ev_data["event_position"] > 1]
    if opener_rows.empty or mid_bout_rows.empty:
        continue
    opener = opener_rows.iloc[0]
    rec = {"event_id": ev_id, "date_folder": ev_data["date_folder"].iloc[0]}
    for _, col, scale in FEATURES_STATE_VIOLIN:
        rec[f"{col}_opener"]  = scale(opener[col])
        rec[f"{col}_midbout"] = scale(mid_bout_rows[col].median())
    records_p.append(rec)
paired_state_full = pd.DataFrame(records_p).dropna()
print(f"Paired events with feature data: {len(paired_state_full)}")
print(f"Per date:\n{paired_state_full.groupby('date_folder').size().to_string()}")

# Plot 3 features x 4 dates.
fig, axes = plt.subplots(
    len(FEATURES_STATE_VIOLIN), len(DATES_TO_PLOT),
    figsize=(2.8 * len(DATES_TO_PLOT), 2.8 * len(FEATURES_STATE_VIOLIN)),
    sharey="row",
)

for r, (ylabel, col, _) in enumerate(FEATURES_STATE_VIOLIN):
    for c, date in enumerate(DATES_TO_PLOT):
        ax = axes[r, c]
        sub = paired_state_full[paired_state_full["date_folder"] == date]
        if sub.empty:
            ax.set_axis_off()
            continue
        openers  = sub[f"{col}_opener"].values
        midbouts = sub[f"{col}_midbout"].values

        parts = ax.violinplot(
            [openers, midbouts], positions=[0, 1],
            widths=0.75, showmeans=False, showmedians=True,
        )
        for pc, color in zip(parts["bodies"], ["#d62728", "#ff7f0e"]):
            pc.set_facecolor(color); pc.set_alpha(0.45)

        for o_val, m_val in zip(openers, midbouts):
            ax.plot([0, 1], [o_val, m_val], "-",
                    color="gray", alpha=0.15, linewidth=0.3, zorder=2)
        ax.scatter([0]*len(openers), openers, s=6,
                   color="#d62728", alpha=0.4, zorder=3, edgecolor="none")
        ax.scatter([1]*len(midbouts), midbouts, s=6,
                   color="#ff7f0e", alpha=0.4, zorder=3, edgecolor="none")

        diff = openers - midbouts
        try:
            _, p = wilcoxon(diff)
        except ValueError:
            p = float("nan")
        med_diff = np.median(diff)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(["event\nopener", "mid-evt\nbout-1"], fontsize=8)
        ax.set_xlim(-0.6, 1.6)
        if c == 0:
            ax.set_ylabel(ylabel, fontsize=9)
        if r == 0:
            ax.set_title(f"{date}  (n={len(sub)})", fontsize=10)
        ax.text(0.5, 0.97,
                f"Δmed = {med_diff:+.2f}\np = {p:.1e} {stars(p)}",
                transform=ax.transAxes, ha="center", va="top", fontsize=8)

fig.suptitle(
    "Event-opener vs mid-event bout-opener  ·  all three features  ·  paired within event",
    y=1.02, fontsize=12,
)
fig.tight_layout()
save_fig(fig, "alarm_violin_state_vs_motor_all_features")
plt.show()


## Spectrogram-pair similarity vs position lag

Theory-free test: does call shape drift continuously through an event?

For every event, compute the log-mel spectrogram of each call (time-normalized
to a fixed frame count so all calls live in the same shape space). Then for
each pair of calls (i, j) in the same event, compute the cosine similarity
between their normalized spectrograms. Plot mean similarity as a function of
position-lag `|j - i|`.

- **Falling curve**: calls are drifting through shape space — adjacent calls
  more similar than far-apart calls. Real continuous gradient.
- **Flat curve**: calls in the same event are equally similar regardless of
  position-gap. No continuous drift.
- **Step at lag = 1 then flat**: the only "drift" is the position-1 vs rest
  step (pairs involving position 1 vs not).

Audio-loading step is ~30 s on the cluster.

In [ ]:
# Spectrogram-pair similarity vs position lag.
# Iterates over events in features_df with >= 3 calls, loads audio per call,
# computes log-mel spec, time-normalizes, and computes pairwise cosines.

import librosa
from scipy.ndimage import zoom
from vocalization_analysis.acoustic_features import load_call_slice

# ---- config ----
N_MEL    = 32       # mel bins
T_FRAMES = 32       # time-normalized frame count
PAD_SEC  = 0.01
MAX_POS_SPEC = 10   # cap calls per event to keep compute tractable
FMIN, FMAX = 1000, 60000

# Merge features_df with calls to get audio-loading coords.
needed_cols = ["bout_id", "bout_position", "exp", "channel", "file_num",
               "start_time_file_sec", "stop_time_file_sec"]
audio_lookup = calls[needed_cols].drop_duplicates(["bout_id", "bout_position"])
fd_audio = features_df.merge(audio_lookup, on=["bout_id", "bout_position"], how="left")
fd_audio = fd_audio.dropna(subset=needed_cols)
print(f"Calls with audio coords: {len(fd_audio):,}")

# Filter to events with >= 3 calls and positions <= MAX_POS_SPEC.
events_to_use = (
    fd_audio[fd_audio["event_position"] <= MAX_POS_SPEC]
    .groupby("event_id")
    .size()
)
events_to_use = events_to_use[events_to_use >= 3].index
print(f"Events with >= 3 calls (positions 1-{MAX_POS_SPEC}): {len(events_to_use)}")

fd_audio = fd_audio[fd_audio["event_id"].isin(events_to_use)].copy()

# ---- Compute log-mel spec per call, time-normalize, L2-normalize ----
def compute_normspec(y, sr):
    S = librosa.feature.melspectrogram(
        y=y.astype(np.float32), sr=sr,
        n_mels=N_MEL, n_fft=512, hop_length=256,
        fmin=FMIN, fmax=min(FMAX, sr // 2),
    )
    S_db = librosa.power_to_db(S + 1e-10, ref=np.max)  # (N_MEL, T)
    if S_db.shape[1] < 2:
        return None
    # Time-normalize to T_FRAMES using zoom on time axis.
    zoom_factor = T_FRAMES / S_db.shape[1]
    S_resized = zoom(S_db, zoom=(1.0, zoom_factor), order=1)
    # Trim/pad to exact T_FRAMES (zoom can be off by one).
    if S_resized.shape[1] > T_FRAMES:
        S_resized = S_resized[:, :T_FRAMES]
    elif S_resized.shape[1] < T_FRAMES:
        S_resized = np.pad(S_resized, ((0, 0),
                                       (0, T_FRAMES - S_resized.shape[1])),
                           mode="edge")
    v = S_resized.flatten()
    v = v - v.mean()
    n = np.linalg.norm(v)
    if n < 1e-8:
        return None
    return v / n

specs_by_call = {}  # (event_id, event_position) -> unit vector
failed = 0
for i, row in enumerate(fd_audio.itertuples(index=False)):
    try:
        y, sr = load_call_slice(
            BASE_PROCESSED_AUDIO,
            row.date_folder, row.exp, row.channel, row.file_num,
            row.start_time_file_sec, row.stop_time_file_sec,
            pad_sec=PAD_SEC,
        )
        v = compute_normspec(y, sr)
        if v is None:
            failed += 1
            continue
        specs_by_call[(row.event_id, row.event_position)] = v
    except Exception:
        failed += 1
    if (i + 1) % 500 == 0:
        print(f"  {i+1:,}/{len(fd_audio):,}")
print(f"Computed specs for {len(specs_by_call):,} calls ({failed} failed)")

# ---- Pairwise similarities within events ----
sim_records = []
for ev_id in events_to_use:
    calls_in_event = sorted(
        [(pos, specs_by_call.get((ev_id, pos)))
         for pos in range(1, MAX_POS_SPEC + 1)
         if (ev_id, pos) in specs_by_call],
        key=lambda t: t[0],
    )
    if len(calls_in_event) < 2:
        continue
    date = fd_audio.loc[fd_audio["event_id"] == ev_id, "date_folder"].iloc[0]
    for a in range(len(calls_in_event)):
        for b in range(a + 1, len(calls_in_event)):
            pos_a, v_a = calls_in_event[a]
            pos_b, v_b = calls_in_event[b]
            sim = float(np.dot(v_a, v_b))
            sim_records.append({
                "event_id": ev_id, "date_folder": date,
                "pos_a": pos_a, "pos_b": pos_b,
                "lag": abs(pos_b - pos_a),
                "involves_pos1": (pos_a == 1) or (pos_b == 1),
                "sim": sim,
            })

sim_df = pd.DataFrame(sim_records)
print(f"\nTotal pairs: {len(sim_df):,}")

# ---- Plot: mean similarity vs lag, two flavors ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: all pairs.
ax = axes[0]
agg_all = sim_df.groupby("lag")["sim"].agg(["mean", "sem", "count"])
ax.errorbar(agg_all.index, agg_all["mean"], yerr=agg_all["sem"],
            marker="o", color="navy", linewidth=1.5, capsize=3, label="all pairs")
ax.set_xlabel("position lag |j - i|")
ax.set_ylabel("mean cosine similarity")
ax.set_title("All pairs")
ax.set_xticks(range(1, MAX_POS_SPEC))
for lag, n in agg_all["count"].items():
    ax.annotate(f"n={int(n)}", (lag, agg_all.loc[lag, "mean"]),
                xytext=(0, -12), textcoords="offset points",
                ha="center", fontsize=7, color="gray")
ax.grid(alpha=0.3)

# Right: split by whether the pair involves position 1.
ax = axes[1]
for involves, color, label in [(True, "tab:red", "pair includes pos-1"),
                                (False, "tab:blue", "pair is pos>=2 vs pos>=2")]:
    sub = sim_df[sim_df["involves_pos1"] == involves]
    if sub.empty:
        continue
    agg = sub.groupby("lag")["sim"].agg(["mean", "sem", "count"])
    ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"],
                marker="o", color=color, linewidth=1.5, capsize=3, label=label)
ax.set_xlabel("position lag |j - i|")
ax.set_ylabel("mean cosine similarity")
ax.set_title("Split by pos-1 involvement")
ax.set_xticks(range(1, MAX_POS_SPEC))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.suptitle(
    f"Spectrogram-pair similarity vs position lag  "
    f"(N_MEL={N_MEL}, T={T_FRAMES}, {len(events_to_use)} events)",
    y=1.02, fontsize=11,
)
fig.tight_layout()
save_fig(fig, "alarm_spec_pair_similarity_vs_lag")
plt.show()

## Adjacent-pair similarity vs position

Complementary view to the similarity-vs-lag plot above:

- **Lag plot** = how *cumulative* drift grows with position-gap (averages over
  pairs anywhere in the event).
- **This plot** = similarity between *adjacent* calls, plotted by where in the
  event the pair sits. x = position of the later call; y = cosine similarity
  with the call at position x-1.

Reads:
- A **dip at x=2** quantifies the position-1 step (call 2 vs the distinct opener).
- A **flat profile from x=3 onward** means drift between adjacent calls is
  uniform (same shape change per step, regardless of where in the event).
- A **rising or falling profile** means drift speed varies through the event.

In [ ]:
# Adjacent-pair similarity vs position-of-later-call.
# Uses the time-normalized specs (specs_by_call) - the version that worked for
# positions 2-15. Includes position 1 here, since the x=2 point is the whole
# point.

adj_records = []
events_set = set(ev for (ev, _) in specs_by_call.keys())

for ev_id in events_set:
    date = date_by_event.get(ev_id) if "date_by_event" in dir() else None
    if date is None:
        m = fd_audio.loc[fd_audio["event_id"] == ev_id, "date_folder"]
        if m.empty:
            continue
        date = m.iloc[0]
    for pos in range(2, MAX_POS_SPEC + 1):
        v_prev = specs_by_call.get((ev_id, pos - 1))
        v_curr = specs_by_call.get((ev_id, pos))
        if v_prev is None or v_curr is None:
            continue
        adj_records.append({
            "event_id":  ev_id,
            "date_folder": date,
            "position":  pos,           # position of the LATER call
            "sim":       float(np.dot(v_prev, v_curr)),
        })

adj_df = pd.DataFrame(adj_records)
print(f"Adjacent pairs: {len(adj_df):,}")
print(f"Events contributing: {adj_df['event_id'].nunique()}")

# ---- Plot: pooled + per-date ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

# Pooled (left).
ax = axes[0]
agg = adj_df.groupby("position")["sim"].agg(["mean", "sem", "count"])
ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"],
            marker="o", color="navy", linewidth=1.5, capsize=3)
for pos, n in agg["count"].items():
    ax.annotate(f"n={int(n)}", (pos, agg.loc[pos, "mean"]),
                xytext=(0, -12), textcoords="offset points",
                ha="center", fontsize=7, color="gray")
ax.set_xlabel("position of later call  (sim with position-1)")
ax.set_ylabel("mean cosine similarity (adjacent pair)")
ax.set_title("Pooled across dates")
ax.set_xticks(range(2, MAX_POS_SPEC + 1))
ax.grid(alpha=0.3)
ax.axvline(2.5, color="red", linestyle=":", linewidth=1, alpha=0.6)

# Per-date (right).
ax = axes[1]
date_colors = {d: c for d, c in zip(
    DATES_TO_PLOT,
    plt.cm.viridis(np.linspace(0.15, 0.85, len(DATES_TO_PLOT)))
)}
for date in DATES_TO_PLOT:
    sub = adj_df[adj_df["date_folder"] == date]
    if sub.empty:
        continue
    agg = sub.groupby("position")["sim"].agg(["mean", "sem"])
    ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"],
                marker="o", color=date_colors[date],
                linewidth=1.5, capsize=3,
                label=f"{date}  (n_evt={sub['event_id'].nunique()})")
ax.set_xlabel("position of later call")
ax.set_title("Per date")
ax.set_xticks(range(2, MAX_POS_SPEC + 1))
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
ax.axvline(2.5, color="red", linestyle=":", linewidth=1, alpha=0.6)

fig.suptitle(
    f"Adjacent-pair spectrogram similarity vs position  "
    f"(time-normalized, {len(specs_by_call):,} calls)",
    y=1.00, fontsize=11,
)
fig.tight_layout()
save_fig(fig, "alarm_spec_adjacent_similarity")
plt.show()

### Within-event slopes: is the stereotyping real, or selection bias?

For each event with at least 4 adjacent-call similarities (i.e., >=5 calls),
fit a simple linear regression `sim ~ position` and keep the slope. Distribution
of per-event slopes:

- **Median > 0, Wilcoxon p << 0.05** -> within events, adjacent calls reliably
  become more similar as the event progresses. Real signal.
- **Median ~ 0** -> the pooled rising curve was selection bias (longer events
  are more stereotyped on average, but no within-event tightening).

This is the artifact-vs-signal test.

In [ ]:
# Within-event slope: adjacent-call similarity ~ position, fit per event.

from scipy.stats import wilcoxon

# Build event -> {pos: vector} once.
event_calls = {}
for (ev_id, pos), v in specs_by_call.items():
    event_calls.setdefault(ev_id, {})[pos] = v

date_by_event = dict(zip(fd_audio["event_id"], fd_audio["date_folder"]))

slope_records = []
for ev_id, calls_here in event_calls.items():
    # Adjacent-pair similarities for this event.
    sims = []
    for p in sorted(calls_here):
        if (p - 1) in calls_here:
            sims.append((p, float(np.dot(calls_here[p - 1], calls_here[p]))))
    if len(sims) < 4:
        continue
    positions = np.array([p for p, _ in sims])
    sim_values = np.array([s for _, s in sims])
    slope, intercept = np.polyfit(positions, sim_values, 1)
    slope_records.append({
        "event_id":    ev_id,
        "date_folder": date_by_event.get(ev_id),
        "slope":       float(slope),
        "intercept":   float(intercept),
        "n_pairs":     len(sims),
    })

slopes_df = pd.DataFrame(slope_records)
slopes_df = slopes_df.dropna(subset=["date_folder", "slope"])
print(f"Events with slope fit: {len(slopes_df)}")
print(f"Median pairs per event: {slopes_df['n_pairs'].median():.0f}")

# Pooled and per-date Wilcoxon vs 0.
med_pool = slopes_df["slope"].median()
stat_pool, p_pool = wilcoxon(slopes_df["slope"], alternative="greater")
print(f"\nPooled: median slope = {med_pool:+.5f},  Wilcoxon (slope > 0) p = {p_pool:.2g}")
for date in DATES_TO_PLOT:
    s = slopes_df.loc[slopes_df["date_folder"] == date, "slope"]
    if len(s) < 3:
        continue
    med_d = s.median()
    stat_d, p_d = wilcoxon(s, alternative="greater")
    print(f"  {date}:  n={len(s):3d}, median = {med_d:+.5f}, "
          f"Wilcoxon p (greater) = {p_d:.2g}")

# ---- Plot: pooled histogram + per-date strip ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axes[0]
ax.hist(slopes_df["slope"], bins=40, color="navy", alpha=0.6, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.7)
ax.axvline(med_pool, color="red", linewidth=1.5, linestyle="--",
           label=f"median = {med_pool:+.5f}")
ax.set_xlabel("per-event slope of adjacent similarity vs position")
ax.set_ylabel("# events")
ax.set_title(
    f"Pooled  (N={len(slopes_df)} events,  Wilcoxon p (>0) = {p_pool:.2g})"
)
ax.legend(fontsize=8)

ax = axes[1]
date_colors = {d: c for d, c in zip(
    DATES_TO_PLOT,
    plt.cm.viridis(np.linspace(0.15, 0.85, len(DATES_TO_PLOT)))
)}
y_jitter = 0.10
for i, date in enumerate(DATES_TO_PLOT):
    s = slopes_df.loc[slopes_df["date_folder"] == date, "slope"]
    if s.empty:
        continue
    rng = np.random.default_rng(i)
    y_pos = i + rng.uniform(-y_jitter, y_jitter, size=len(s))
    ax.scatter(s, y_pos, color=date_colors[date], alpha=0.6, s=18)
    ax.scatter(s.median(), i, color="black", marker="|", s=200, linewidth=2)
ax.axvline(0, color="black", linewidth=0.7)
ax.set_yticks(range(len(DATES_TO_PLOT)))
ax.set_yticklabels(DATES_TO_PLOT)
ax.set_xlabel("per-event slope")
ax.set_title("Per date (black bar = median)")
ax.grid(axis="x", alpha=0.3)

fig.suptitle("Within-event slope of adjacent-call similarity vs position",
             y=1.00, fontsize=11)
fig.tight_layout()
save_fig(fig, "alarm_spec_adjacent_slope_per_event")
plt.show()

### Distance-to-late-template: how close is each call to the "settled" shape?

Same idea as the LOO event-template plot, but the template is now defined as
the mean of **late-position calls only** (positions >= `LATE_THRESHOLD`,
default = 5). This is the cleanest test of "stereotypy at the end":

- **Late calls** (positions >= 5) are close to the template by construction
  (they define it). Leave-one-out so we don't double-count the call being
  scored.
- **Early calls** (positions 1-4) are *not* in the template - we ask how far
  each of them is from the settled-shape direction.

Expected pattern: rising line through the early positions, plateau at high
positions. Sharp drop only at position 1 = strong "opener" signal.

In [ ]:
# Distance to late-call template, per call.

LATE_THRESHOLD = 5

late_tmpl_records = []
for ev_id, calls_here in event_calls.items():
    late_positions = [p for p in calls_here if p >= LATE_THRESHOLD]
    # Need at least 2 late calls so the LOO template still has >= 1 call.
    if len(late_positions) < 2:
        continue
    for p in sorted(calls_here):
        # LOO: drop the call being measured (only relevant if p itself is late).
        late_subset = [calls_here[q] for q in late_positions if q != p]
        if not late_subset:
            continue
        template = np.mean(late_subset, axis=0)
        n_t = np.linalg.norm(template)
        if n_t < 1e-8:
            continue
        sim = float(np.dot(calls_here[p], template) / n_t)
        late_tmpl_records.append({
            "event_id":      ev_id,
            "date_folder":   date_by_event.get(ev_id),
            "position":      p,
            "sim_to_late":   sim,
            "is_late_call":  p >= LATE_THRESHOLD,
        })

late_df = pd.DataFrame(late_tmpl_records)
print(f"Calls with late-template sim: {len(late_df):,}")
print(f"Events contributing (have >= 2 calls at positions >= {LATE_THRESHOLD}): "
      f"{late_df['event_id'].nunique()}")

# ---- Plot: pooled + per-date ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

ax = axes[0]
sub = late_df[late_df["position"] <= MAX_POS_SPEC]
agg = sub.groupby("position")["sim_to_late"].agg(["mean", "sem", "count"])
ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"],
            marker="o", color="darkred", linewidth=1.5, capsize=3)
ax.axvline(LATE_THRESHOLD - 0.5, color="black", linestyle=":",
           linewidth=1, alpha=0.6,
           label=f"late threshold (>= {LATE_THRESHOLD})")
for pos, n in agg["count"].items():
    ax.annotate(f"n={int(n)}", (pos, agg.loc[pos, "mean"]),
                xytext=(0, -12), textcoords="offset points",
                ha="center", fontsize=7, color="gray")
ax.set_xlabel("position in event")
ax.set_ylabel("cosine sim to late-call template")
ax.set_title("Pooled across dates")
ax.set_xticks(range(1, MAX_POS_SPEC + 1))
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)

ax = axes[1]
date_colors = {d: c for d, c in zip(
    DATES_TO_PLOT,
    plt.cm.viridis(np.linspace(0.15, 0.85, len(DATES_TO_PLOT)))
)}
for date in DATES_TO_PLOT:
    sub = late_df[(late_df["date_folder"] == date)
                  & (late_df["position"] <= MAX_POS_SPEC)]
    if sub.empty:
        continue
    agg = sub.groupby("position")["sim_to_late"].agg(["mean", "sem"])
    ax.errorbar(agg.index, agg["mean"], yerr=agg["sem"],
                marker="o", color=date_colors[date], linewidth=1.5, capsize=3,
                label=f"{date}  (n_evt={sub['event_id'].nunique()})")
ax.axvline(LATE_THRESHOLD - 0.5, color="black", linestyle=":",
           linewidth=1, alpha=0.6)
ax.set_xlabel("position in event")
ax.set_title("Per date")
ax.set_xticks(range(1, MAX_POS_SPEC + 1))
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)

fig.suptitle(
    f"Each call's similarity to its event's LATE-call template  "
    f"(positions >= {LATE_THRESHOLD})",
    y=1.00, fontsize=11,
)
fig.tight_layout()
save_fig(fig, "alarm_spec_distance_to_late_template")
plt.show()

## Bulk-export bout spectrogram pages — moved to a standalone script

The in-kernel version kept crashing on long runs. Moved to:

```
scripts/export_bout_pages.py
```

Run from the cluster shell (clean process, no kernel state to leak):

```bash
cd /mnt/home/gginosar/repos/gerbil_vocalization_analysis
python scripts/export_bout_pages.py
```

What it does:
- 4 sheets per date, 20 bouts per sheet (chronologically ordered + evenly sampled).
- **Brighter spectrograms** (vmin=-80, vmax=0, magma) so faint structure shows.
- **Short vertical tick** at each call's start time only (no duration line).
- **ICI in ms labelled above each tick** (next to the indication line).
- **X-ticks every 1 s**, x-label on bottom row only.
- Outputs to `BASE_PROCESSED_AUDIO/alarm/bout_example_pages/` as `alarm_bout_examples_{date}_sheet{N}.png`.

Knobs in the file's config block (lines ~40-55): `N_PER_PAGE`, `SHEETS_PER_DATE`, `MIN_BOUT_SIZE`, `MAX_BOUT_DUR_S`, `CMAP`, `VMIN/VMAX`, etc. Will reuse this script for the other talk-figure types later (just add functions to it).

In [ ]:
# This used to run the bulk PNG export in-kernel; it kept crashing.
# The code now lives at scripts/export_bout_pages.py - run it from a shell:
#
#   cd /mnt/home/gginosar/repos/gerbil_vocalization_analysis
#   python scripts/export_bout_pages.py
#
# To render a subset of dates only:
#   python scripts/export_bout_pages.py --dates 2025_03 2025_07
#
# To redirect the output dir:
#   python scripts/export_bout_pages.py --out-dir /some/other/path

import subprocess, sys
print("To regenerate the bout-example sheets, run from a shell:")
print("  python scripts/export_bout_pages.py")
print("(see the markdown cell above for options)")
